In [2]:
!pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.9 MB/s eta 0:00:00


In [3]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")
print("Model:", model_name)
print("Device:", model.device)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully!
Model: Qwen/Qwen3-1.7B
Device: cuda:0


In [4]:

prompt = """
You are reviewing a mathematical proof.

Consider f_n(x) = x^n on [0, 1].
A student claims:

"Since f_n(x) converges pointwise for every x in [0, 1],
the sequence converges uniformly on [0, 1]."

Is the student's conclusion correct?
Explain your mathematical reasoning carefully.
At the end, state your verbal confidence as HIGH, MEDIUM, or LOW,
and briefly explain why.

"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("MODEL:", model_name)
print("=" * 60)
print(answer)


MODEL: Qwen/Qwen3-1.7B
Let's analyze the student's claim step by step.

---

### **Given:**

- $ f_n(x) = x^n $ on $[0, 1]$
- The student claims: **"Since $ f_n(x) $ converges pointwise for every $ x \in [0, 1] $, the sequence converges uniformly on $[0, 1]$."**

---

### **Step 1: Pointwise Convergence**

We first check whether $ f_n(x) $ converges **pointwise** on $[0, 1]$.

- For $ x \in [0, 1) $: $ x^n \to 0 $ as $ n \to \infty $, because $ x < 1 $ and raising it to higher powers makes it smaller.
- For $ x = 1 $: $ x^n = 1^n = 1 $, so $ f_n(1) = 1 $ for all $ n $.

So, the pointwise limit function is:
$$
f(x) = \begin{cases}
0 & \text{if } 0 \le x < 1 \\
1 & \text{if } x = 1
\end{cases}
$$

This is a **discontinuous** function at $ x = 1 $, but it is still a valid pointwise limit.

---

### **Step 2: Uniform Convergence**

Now, the student claims that the sequence converges **uniformly** on $[0, 1]$.

But **uniform convergence** is a stronger condition than pointwise convergence. 

In [5]:

print(answer[-3000:])


Let's analyze the student's claim step by step.

---

### **Given:**

- $ f_n(x) = x^n $ on $[0, 1]$
- The student claims: **"Since $ f_n(x) $ converges pointwise for every $ x \in [0, 1] $, the sequence converges uniformly on $[0, 1]$."**

---

### **Step 1: Pointwise Convergence**

We first check whether $ f_n(x) $ converges **pointwise** on $[0, 1]$.

- For $ x \in [0, 1) $: $ x^n \to 0 $ as $ n \to \infty $, because $ x < 1 $ and raising it to higher powers makes it smaller.
- For $ x = 1 $: $ x^n = 1^n = 1 $, so $ f_n(1) = 1 $ for all $ n $.

So, the pointwise limit function is:
$$
f(x) = \begin{cases}
0 & \text{if } 0 \le x < 1 \\
1 & \text{if } x = 1
\end{cases}
$$

This is a **discontinuous** function at $ x = 1 $, but it is still a valid pointwise limit.

---

### **Step 2: Uniform Convergence**

Now, the student claims that the sequence converges **uniformly** on $[0, 1]$.

But **uniform convergence** is a stronger condition than pointwise convergence. It requires that the di

In [6]:

prompt = """For f_n(x)=x^n on [0,1], a student claims:
Pointwise convergence implies uniform convergence.

Reply in this exact format:
Verdict: CORRECT or INCORRECT
Reason: (maximum 3 sentences, with the key mathematical calculation)
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: (one sentence)
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False
    )

answer_short = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer_short)


Verdict: INCORRECT  
Reason: Pointwise convergence does not imply uniform convergence because, for example, the sequence $ f_n(x) = x^n $ converges pointwise to 0 on [0,1], but does not converge uniformly.  
Confidence: MEDIUM  
Why this confidence: The uniform convergence would require the maximum difference between $ f_n(x) $ and 0 to approach 0 uniformly across all $ x \in [0,1] $, which is not the case here.


In [7]:

prompt = r"""
Consider the following proposed proof:

A sequence of functions f_n converges uniformly to f on a set E
because, for every epsilon > 0 and every x in E, there exists
N depending on x such that for all n >= N,
|f_n(x) - f(x)| < epsilon.

Evaluate the proof.

Reply in exactly this format:
Verdict: CORRECT or INCORRECT
Reason: (maximum 3 sentences; explain the quantifier issue)
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: (one sentence)
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False
    )

answer_case2 = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer_case2)


Verdict: INCORRECT  
Reason: The proof incorrectly states that a sequence of functions converges uniformly to f on E because for every x in E, there exists N depending on x such that for all n ≥ N, |f_n(x) - f(x)| < ε. The correct definition of uniform convergence requires that for every ε > 0, there exists N such that for all x in E and all n ≥ N, |f_n(x) - f(x)| < ε, regardless of x.  
Confidence: LOW  
Why this confidence: The proof fails to address the universal quantifier over x and n, which is essential for uniform convergence.


In [8]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Tokenizer ready:", tokenizer is not None)
print("Model device:", model.device)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Tokenizer ready: True
Model device: cuda:0


In [9]:

prompt = r"""
Consider the following proof:

Let f_n(x) = x/n on E = [0,1], and let f(x) = 0.
For every x in [0,1],
|f_n(x) - f(x)| = x/n <= 1/n.

Given any epsilon > 0, choose an integer N > 1/epsilon.
Then for every n >= N and every x in [0,1],
|f_n(x) - f(x)| <= 1/n <= 1/N < epsilon.

Therefore, f_n converges uniformly to f on [0,1].

Evaluate the proof.

Reply in exactly this format:
Verdict: CORRECT or INCORRECT
Reason: (maximum 3 sentences)
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: (one sentence)
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False
    )

answer_case3 = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer_case3)


Verdict: INCORRECT  
Reason: The proof is incorrect because it assumes that $ \frac{1}{N} < \epsilon $ for some $ N > \frac{1}{\epsilon} $, but this is only valid if $ \epsilon $ is fixed and $ N $ is chosen accordingly. The error lies in the choice of $ N $, as $ N $ should be chosen such that $ \frac{1}{N} < \epsilon $, but the proof does not properly establish this for all $ \epsilon $.  
Confidence: MEDIUM  
Why this confidence: The proof incorrectly assumes that $ \frac{1}{N} < \epsilon $ for some $ N $, but the correct approach requires choosing $ N $ such that $ \frac{1}{N} < \epsilon $, which is not properly justified in the given proof.


In [10]:

prompt = r"""
Let f_n(x)=x/n on [0,1] and f(x)=0.
For every epsilon>0, choose an integer N>1/epsilon.
For all n>=N and all x in [0,1],
|f_n(x)-f(x)|=x/n<=1/n<=1/N<epsilon.
Thus f_n converges uniformly to zero.

Is this proof valid?
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW
"""

messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=120, do_sample=False
    )

answer_case4 = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)
print(answer_case4)


Verdict: CORRECT  
Reason: The proof correctly applies the uniform convergence criterion by showing that for all n ≥ N, the difference |f_n(x) - f(x)| = x/n ≤ 1/n ≤ 1/N < ε, hence proving uniform convergence to zero.


In [11]:

print("Model exists:", "model" in globals() and model is not None)
print("Tokenizer exists:", "tokenizer" in globals() and tokenizer is not None)

if "model" in globals() and model is not None:
    print("Model device:", next(model.parameters()).device)


Model exists: True
Tokenizer exists: True
Model device: cuda:0


In [12]:

prompt = r"""
Let f_n(x)=x/n on [0,1] and f(x)=0.
For every epsilon>0, choose an integer N>1/epsilon.
For all n>=N and all x in [0,1],
|f_n(x)-f(x)|=x/n<=1/n<=1/N<epsilon.
Thus f_n converges uniformly to zero.

Is this proof valid?
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

answer_case4 = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer_case4)


Verdict: CORRECT  
Reason: The proof correctly applies the uniform convergence criterion by showing that for all n ≥ N, the difference |f_n(x) - f(x)| = x/n ≤ 1/n ≤ 1/N < ε, hence proving uniform convergence to zero.


In [13]:

# Fatima Fellowship — Candidate Test Cases
# This cell only prepares and displays reference cases.
# It does NOT run the model or modify previous results.

test_cases = [
    {
        "id": "case5",
        "topic": "Uniform convergence on a smaller interval",
        "prompt": r"""
Let f_n(x) = x^n on [0, 1/2], and let f(x) = 0.
Since 0 <= x <= 1/2, we have |f_n(x)-f(x)| = x^n <= (1/2)^n.
For every epsilon > 0, choose N such that (1/2)^N < epsilon.
Then for every n >= N and every x in [0, 1/2],
|f_n(x)-f(x)| <= (1/2)^n <= (1/2)^N < epsilon.
Therefore, f_n converges uniformly to zero.

Is this proof valid?
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW
""",
        "reference_verdict": "CORRECT",
        "reference_reason": (
            "The bound |f_n(x)| <= (1/2)^n is uniform in x, "
            "and (1/2)^n tends to zero. An integer N can be "
            "chosen sufficiently large for every epsilon > 0."
        )
    },
    {
        "id": "case6",
        "topic": "Pointwise but not uniform convergence on [0, 1)",
        "prompt": r"""
Let f_n(x) = x^n on [0, 1), and let f(x) = 0.
For every fixed x in [0, 1), x^n tends to zero.
A student concludes that f_n converges uniformly to zero on [0, 1).

Is the student's conclusion correct?
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW
""",
        "reference_verdict": "INCORRECT",
        "reference_reason": (
            "Although convergence is pointwise, the supremum "
            "of x^n over x in [0, 1) equals 1 for every n. "
            "Thus the supremum does not tend to zero."
        )
    },
    {
        "id": "case7",
        "topic": "Pointwise convergence on the real line",
        "prompt": r"""
Let f_n(x) = sin(x/n) for all real x, and let f(x) = 0.
For every fixed real x, sin(x/n) tends to zero as n tends to infinity.
A student concludes that f_n converges uniformly to zero on R.

Is the student's conclusion correct?
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW
""",
        "reference_verdict": "INCORRECT",
        "reference_reason": (
            "For every positive integer n, the supremum over "
            "all real x of |sin(x/n)| equals 1. Therefore, "
            "the supremum does not tend to zero, so convergence "
            "is not uniform on R."
        )
    }
]

for case in test_cases:
    print("=" * 70)
    print("ID:", case["id"])
    print("TOPIC:", case["topic"])
    print("\nPROMPT:")
    print(case["prompt"])
    print("\nREFERENCE VERDICT:", case["reference_verdict"])
    print("REFERENCE REASON:", case["reference_reason"])
    print()

print("Total candidate cases:", len(test_cases))
print("No model inference was performed.")


ID: case5
TOPIC: Uniform convergence on a smaller interval

PROMPT:

Let f_n(x) = x^n on [0, 1/2], and let f(x) = 0.
Since 0 <= x <= 1/2, we have |f_n(x)-f(x)| = x^n <= (1/2)^n.
For every epsilon > 0, choose N such that (1/2)^N < epsilon.
Then for every n >= N and every x in [0, 1/2],
|f_n(x)-f(x)| <= (1/2)^n <= (1/2)^N < epsilon.
Therefore, f_n converges uniformly to zero.

Is this proof valid?
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW


REFERENCE VERDICT: CORRECT
REFERENCE REASON: The bound |f_n(x)| <= (1/2)^n is uniform in x, and (1/2)^n tends to zero. An integer N can be chosen sufficiently large for every epsilon > 0.

ID: case6
TOPIC: Pointwise but not uniform convergence on [0, 1)

PROMPT:

Let f_n(x) = x^n on [0, 1), and let f(x) = 0.
For every fixed x in [0, 1), x^n tends to zero.
A student concludes that f_n converges uniformly to zero on [0, 1).

Is the student's conclusion correct?
Reply with:
Verdict: CORRECT or 

In [14]:

# Fatima Fellowship — Run Cases 5–7
# Uses the existing model and tokenizer.
# Keeps the original prompts and saves the raw generated answers.

import torch

assert "model" in globals() and model is not None, "Model is not loaded."
assert "tokenizer" in globals() and tokenizer is not None, "Tokenizer is not loaded."
assert "test_cases" in globals(), "Run the candidate-case preparation cell first."

case_results = {}

for case in test_cases:
    case_id = case["id"]

    messages = [{"role": "user", "content": case["prompt"]}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    raw_answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    case_results[case_id] = raw_answer

    print("=" * 70)
    print("CASE:", case_id)
    print("MODEL:", "Qwen/Qwen3-1.7B")
    print("RAW MODEL OUTPUT:")
    print(raw_answer)
    print()

print("Completed model runs:", len(case_results))


CASE: case5
MODEL: Qwen/Qwen3-1.7B
RAW MODEL OUTPUT:
Verdict: CORRECT  
Reason: The proof correctly applies the uniform convergence criterion, showing that the sequence $ f_n(x) = x^n $ converges uniformly to 0 on $[0, 1/2]$ by bounding the difference by $ (1/2)^n $, which tends to 0 as $ n \to \infty $.

CASE: case6
MODEL: Qwen/Qwen3-1.7B
RAW MODEL OUTPUT:
Verdict: INCORRECT  
Reason: The student's conclusion is incorrect because uniform convergence requires that the maximum of |f_n(x) - f(x)| tends to zero uniformly over the entire interval. Here, f_n(x) = x^n on [0, 1), and the maximum of |x^n| occurs at x = 1, but 1^n = 1, not 0. Thus, the maximum does not tend to zero.

CASE: case7
MODEL: Qwen/Qwen3-1.7B
RAW MODEL OUTPUT:
Verdict: INCORRECT  
Reason: The student's conclusion is incorrect because uniform convergence requires that the maximum of |f_n(x) - f(x)| tends to zero uniformly over all x. Since f_n(x) = sin(x/n), the maximum value of |sin(x/n)| is 1, and it does not tend to 

In [15]:

import csv
import json
from pathlib import Path

# Fatima Fellowship — Experimental Results Log
# No model inference is performed in this cell.

results = [
    {
        "case_id": "case1",
        "topic": "Pointwise versus uniform convergence on [0,1]",
        "reference_verdict": "INCORRECT",
        "model_verdict": "INCORRECT",
        "verdict_correct": True,
        "reason_assessment": "PARTIALLY_INCORRECT",
        "verbal_confidence": "MEDIUM",
        "raw_output": None,
        "notes": (
            "The recorded explanation incorrectly stated that x^n "
            "converges pointwise to zero on all of [0,1], overlooking x=1. "
            "The exact complete raw output is not preserved in this log."
        ),
    },
    {
        "case_id": "case2",
        "topic": "Incorrect dependence of N on x",
        "reference_verdict": "INCORRECT",
        "model_verdict": "INCORRECT",
        "verdict_correct": True,
        "reason_assessment": "CORRECT",
        "verbal_confidence": "LOW",
        "raw_output": None,
        "notes": (
            "The recorded assessment says the model correctly identified "
            "that N must work independently of x. Exact raw output not preserved."
        ),
    },
    {
        "case_id": "case3",
        "topic": "Valid uniform convergence proof for f_n(x)=x/n",
        "reference_verdict": "CORRECT",
        "model_verdict": "INCORRECT",
        "verdict_correct": False,
        "reason_assessment": "INCORRECT",
        "verbal_confidence": "MEDIUM",
        "raw_output": None,
        "notes": (
            "The model reportedly rejected the valid choice N > 1/epsilon. "
            "Exact raw output not preserved."
        ),
    },
    {
        "case_id": "case4",
        "topic": "Paraphrase of the valid x/n proof",
        "reference_verdict": "CORRECT",
        "model_verdict": "CORRECT",
        "verdict_correct": True,
        "reason_assessment": "CORRECT",
        "verbal_confidence": None,
        "raw_output": (
            "Verdict: CORRECT\n"
            "Reason: The proof correctly applies the uniform convergence "
            "criterion by showing that for all n ≥ N, the difference "
            "|f_n(x) - f(x)| = x/n ≤ 1/n ≤ 1/N < ε, hence proving "
            "uniform convergence to zero."
        ),
        "notes": (
            "The model did not provide the requested confidence label. "
            "The prompt was shorter than the Case 3 prompt."
        ),
    },
    {
        "case_id": "case5",
        "topic": "Uniform convergence on [0,1/2]",
        "reference_verdict": "CORRECT",
        "model_verdict": "CORRECT",
        "verdict_correct": True,
        "reason_assessment": "CORRECT",
        "verbal_confidence": None,
        "raw_output": (
            "Verdict: CORRECT  \n"
            "Reason: The proof correctly applies the uniform convergence "
            "criterion, showing that the sequence $ f_n(x) = x^n $ "
            "converges uniformly to 0 on $[0, 1/2]$ by bounding the "
            "difference by $ (1/2)^n $, which tends to 0 as $ n \\to \\infty $."
        ),
        "notes": "Confidence label not provided by the model.",
    },
    {
        "case_id": "case6",
        "topic": "Pointwise but not uniform convergence on [0,1)",
        "reference_verdict": "INCORRECT",
        "model_verdict": "INCORRECT",
        "verdict_correct": True,
        "reason_assessment": "PARTIALLY_INCORRECT",
        "verbal_confidence": None,
        "raw_output": (
            "Verdict: INCORRECT  \n"
            "Reason: The student's conclusion is incorrect because uniform "
            "convergence requires that the maximum of |f_n(x) - f(x)| "
            "tends to zero uniformly over the entire interval. Here, "
            "f_n(x) = x^n on [0, 1), and the maximum of |x^n| occurs at "
            "x = 1, but 1^n = 1, not 0. Thus, the maximum does not tend to zero."
        ),
        "notes": (
            "The verdict is correct, but the explanation incorrectly says "
            "the maximum occurs at x=1, which is outside the domain. "
            "The supremum is 1."
        ),
    },
    {
        "case_id": "case7",
        "topic": "Pointwise convergence of sin(x/n) on R",
        "reference_verdict": "INCORRECT",
        "model_verdict": "INCORRECT",
        "verdict_correct": True,
        "reason_assessment": "CORRECT",
        "verbal_confidence": None,
        "raw_output": (
            "Verdict: INCORRECT  \n"
            "Reason: The student's conclusion is incorrect because uniform "
            "convergence requires that the maximum of |f_n(x) - f(x)| "
            "tends to zero uniformly over all x. Since f_n(x) = sin(x/n), "
            "the maximum value of |sin(x/n)| is 1, and it does not tend "
            "to zero uniformly."
        ),
        "notes": "Confidence label not provided by the model.",
    },
]

# Mark records whose exact original output is unavailable.
for record in results:
    record["raw_output_status"] = (
        "NOT_PRESERVED"
        if record["raw_output"] is None
        else "RECORDED"
    )

out_dir = Path("/content")
csv_path = out_dir / "fatima_fellowship_results.csv"
json_path = out_dir / "fatima_fellowship_results.json"

with csv_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

with json_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("CSV saved:", csv_path)
print("JSON saved:", json_path)
print("Total logged cases:", len(results))
print(
    "Exact outputs recorded:",
    sum(r["raw_output"] is not None for r in results)
)
print(
    "Exact outputs not preserved:",
    sum(r["raw_output"] is None for r in results)
)

print("\nFiles created:")
print(csv_path.exists(), csv_path.stat().st_size, "bytes")
print(json_path.exists(), json_path.stat().st_size, "bytes")


CSV saved: /content/fatima_fellowship_results.csv
JSON saved: /content/fatima_fellowship_results.json
Total logged cases: 7
Exact outputs recorded: 4
Exact outputs not preserved: 3

Files created:
True 2634 bytes
True 4167 bytes


In [16]:
import pandas as pd
import json
from pathlib import Path

csv_path = Path("/content/fatima_fellowship_results.csv")
json_path = Path("/content/fatima_fellowship_results.json")

# Read both files
df = pd.read_csv(csv_path)

with open(json_path, "r", encoding="utf-8") as f:
    records = json.load(f)

print("CSV rows:", len(df))
print("JSON records:", len(records))

print("\nCSV columns:")
print(df.columns.tolist())

print("\nRaw output status:")
if "raw_output_status" in df.columns:
    print(df["raw_output_status"].value_counts(dropna=False))
else:
    print("Column not found")

print("\nCases overview:")
display(df)

print("\nValidation:")
print("CSV exists:", csv_path.exists())
print("JSON exists:", json_path.exists())
print("Case IDs unique:", df["case_id"].is_unique if "case_id" in df.columns else "Unknown")

CSV rows: 7
JSON records: 7

CSV columns:
['case_id', 'topic', 'reference_verdict', 'model_verdict', 'verdict_correct', 'reason_assessment', 'verbal_confidence', 'raw_output', 'notes', 'raw_output_status']

Raw output status:
raw_output_status
RECORDED         4
NOT_PRESERVED    3
Name: count, dtype: int64

Cases overview:


,case_id,topic,reference_verdict,model_verdict,verdict_correct,reason_assessment,verbal_confidence,raw_output,notes,raw_output_status
0,case1,"Pointwise versus uniform convergence on [0,1]",INCORRECT,INCORRECT,True,PARTIALLY_INCORRECT,MEDIUM,NaN,The recorded explanation incorrectly stated th...,NOT_PRESERVED
1,case2,Incorrect dependence of N on x,INCORRECT,INCORRECT,True,CORRECT,LOW,NaN,The recorded assessment says the model correct...,NOT_PRESERVED
2,case3,Valid uniform convergence proof for f_n(x)=x/n,CORRECT,INCORRECT,False,INCORRECT,MEDIUM,NaN,The model reportedly rejected the valid choice...,NOT_PRESERVED
3,case4,Paraphrase of the valid x/n proof,CORRECT,CORRECT,True,CORRECT,NaN,Verdict: CORRECT\nReason: The proof correctly ...,The model did not provide the requested confid...,RECORDED
4,case5,"Uniform convergence on [0,1/2]",CORRECT,CORRECT,True,CORRECT,NaN,Verdict: CORRECT \nReason: The proof correctl...,Confidence label not provided by the model.,RECORDED
5,case6,"Pointwise but not uniform convergence on [0,1)",INCORRECT,INCORRECT,True,PARTIALLY_INCORRECT,NaN,Verdict: INCORRECT \nReason: The student's co...,"The verdict is correct, but the explanation in...",RECORDED
6,case7,Pointwise convergence of sin(x/n) on R,INCORRECT,INCORRECT,True,CORRECT,NaN,Verdict: INCORRECT \nReason: The student's co...,Confidence label not provided by the model.,RECORDED



Validation:
CSV exists: True
JSON exists: True
Case IDs unique: True


In [17]:
for name in ["answer", "answer_short", "answer_case2", "answer_case3"]:
    print("\n" + "=" * 60)
    print("VARIABLE:", name)

    if name in globals():
        print(globals()[name])
    else:
        print("it is note in the Memory right now")



VARIABLE: answer
Let's analyze the student's claim step by step.

---

### **Given:**

- $ f_n(x) = x^n $ on $[0, 1]$
- The student claims: **"Since $ f_n(x) $ converges pointwise for every $ x \in [0, 1] $, the sequence converges uniformly on $[0, 1]$."**

---

### **Step 1: Pointwise Convergence**

We first check whether $ f_n(x) $ converges **pointwise** on $[0, 1]$.

- For $ x \in [0, 1) $: $ x^n \to 0 $ as $ n \to \infty $, because $ x < 1 $ and raising it to higher powers makes it smaller.
- For $ x = 1 $: $ x^n = 1^n = 1 $, so $ f_n(1) = 1 $ for all $ n $.

So, the pointwise limit function is:
$$
f(x) = \begin{cases}
0 & \text{if } 0 \le x < 1 \\
1 & \text{if } x = 1
\end{cases}
$$

This is a **discontinuous** function at $ x = 1 $, but it is still a valid pointwise limit.

---

### **Step 2: Uniform Convergence**

Now, the student claims that the sequence converges **uniformly** on $[0, 1]$.

But **uniform convergence** is a stronger condition than pointwise convergence. It re

In [18]:

import json
import csv
from pathlib import Path

out_dir = Path("/content")
json_path = out_dir / "fatima_fellowship_results.json"
csv_path = out_dir / "fatima_fellowship_results.csv"

# Check that the original answers are still available
required = ["answer", "answer_short", "answer_case2", "answer_case3"]
missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"Missing variables: {missing}")

# Preserve each generated answer separately
recovered_outputs = [
    {
        "output_id": "case1_long",
        "case_id": "case1",
        "variable": "answer",
        "raw_output": answer,
        "note": "Original long response; kept separately from the short-format response."
    },
    {
        "output_id": "case1_short",
        "case_id": "case1",
        "variable": "answer_short",
        "raw_output": answer_short,
        "note": "Short-format response used for the existing Case 1 assessment."
    },
    {
        "output_id": "case2",
        "case_id": "case2",
        "variable": "answer_case2",
        "raw_output": answer_case2,
        "note": "Recovered original model response."
    },
    {
        "output_id": "case3",
        "case_id": "case3",
        "variable": "answer_case3",
        "raw_output": answer_case3,
        "note": "Recovered original model response."
    }
]

archive_path = out_dir / "fatima_fellowship_recovered_outputs.json"

with archive_path.open("w", encoding="utf-8") as f:
    json.dump(recovered_outputs, f, ensure_ascii=False, indent=2)

# Update the main results file
with json_path.open("r", encoding="utf-8") as f:
    results = json.load(f)

answers_for_main_log = {
    "case1": answer_short,
    "case2": answer_case2,
    "case3": answer_case3
}

for record in results:
    case_id = record["case_id"]

    if case_id in answers_for_main_log:
        record["raw_output"] = answers_for_main_log[case_id]
        record["raw_output_status"] = "RECORDED"

    if case_id == "case1":
        record["notes"] += (
            " The long-form output is preserved separately in "
            "fatima_fellowship_recovered_outputs.json."
        )

with json_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with csv_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("Updated main JSON:", json_path.exists())
print("Updated main CSV:", csv_path.exists())
print("Recovered outputs archive:", archive_path.exists())
print("Recovered original outputs:", len(recovered_outputs))
print("Cases with recovered raw outputs:", sum(
    r["raw_output_status"] == "RECORDED" for r in results
))


Updated main JSON: True
Updated main CSV: True
Recovered outputs archive: True
Recovered original outputs: 4
Cases with recovered raw outputs: 7


In [19]:
from google.colab import files

files.download("/content/fatima_fellowship_results.csv")
files.download("/content/fatima_fellowship_results.json")
files.download("/content/fatima_fellowship_recovered_outputs.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import json
from pathlib import Path

path = Path("/content/fatima_fellowship_results.json")

with path.open("r", encoding="utf-8") as f:
    results = json.load(f)

print("Total cases:", len(results))

print("\nCase ID | Verdict | Raw output saved")
for r in results:
    print(
        r["case_id"],
        "|", r["model_verdict"],
        "|", bool(r.get("raw_output"))
    )

print("\nCases with missing raw outputs:")
for r in results:
    if not r.get("raw_output"):
        print(r["case_id"])

print("\nConfidence labels:")
for r in results:
    print(r["case_id"], ":", r.get("verbal_confidence"))

Total cases: 7

Case ID | Verdict | Raw output saved
case1 | INCORRECT | True
case2 | INCORRECT | True
case3 | INCORRECT | True
case4 | CORRECT | True
case5 | CORRECT | True
case6 | INCORRECT | True
case7 | INCORRECT | True

Cases with missing raw outputs:

Confidence labels:
case1 : MEDIUM
case2 : LOW
case3 : MEDIUM
case4 : None
case5 : None
case6 : None
case7 : None


In [21]:

import json
from pathlib import Path

out_dir = Path("/content")

# Load the current results and recovered original outputs
with open(out_dir / "fatima_fellowship_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

with open(out_dir / "fatima_fellowship_recovered_outputs.json", "r", encoding="utf-8") as f:
    recovered = json.load(f)

# Prompts used in the experiment.
# Cases 5–7 are taken from the candidate cases already prepared in the notebook.
prompts = {
    "case1": {
        "prompt_type": "short_format",
        "prompt": """For f_n(x)=x^n on [0,1], a student claims:
Pointwise convergence implies uniform convergence.

Reply in this exact format:
Verdict: CORRECT or INCORRECT
Reason: (maximum 3 sentences, with the key mathematical calculation)
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: (one sentence)"""
    },
    "case2": {
        "prompt_type": "original",
        "prompt": """Consider the following proposed proof:

A sequence of functions f_n converges uniformly to f on a set E
because, for every epsilon > 0 and every x in E, there exists
N depending on x such that for all n >= N,
|f_n(x) - f(x)| < epsilon.

Evaluate the proof.

Reply in exactly this format:
Verdict: CORRECT or INCORRECT
Reason: (maximum 3 sentences; explain the quantifier issue)
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: (one sentence)"""
    },
    "case3": {
        "prompt_type": "original",
        "prompt": """Consider the following proof:

Let f_n(x) = x/n on E = [0,1], and let f(x) = 0.
For every x in [0,1],
|f_n(x) - f(x)| = x/n <= 1/n.

Given any epsilon > 0, choose an integer N > 1/epsilon.
Then for every n >= N and every x in [0,1],
|f_n(x) - f(x)| <= 1/n <= 1/N < epsilon.

Therefore, f_n converges uniformly to f on [0,1].

Evaluate the proof.

Reply in exactly this format:
Verdict: CORRECT or INCORRECT
Reason: (maximum 3 sentences)
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: (one sentence)"""
    },
    "case4": {
        "prompt_type": "paraphrase",
        "prompt": """Let f_n(x)=x/n on [0,1] and f(x)=0.
For every epsilon>0, choose an integer N>1/epsilon.
For all n>=N and all x in [0,1],
|f_n(x)-f(x)|=x/n<=1/n<=1/N<epsilon.
Thus f_n converges uniformly to zero.

Is this proof valid?
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW"""
    }
}

# Reuse the actual prompts stored in the notebook for cases 5–7.
for case in test_cases:
    prompts[case["id"]] = {
        "prompt_type": "candidate_case",
        "prompt": case["prompt"]
    }

# Build one record per test case.
dataset = []

for result in results:
    case_id = result["case_id"]
    if case_id not in prompts:
        raise ValueError(f"Prompt not found for {case_id}")

    dataset.append({
        **result,
        "prompt": prompts[case_id]["prompt"],
        "prompt_type": prompts[case_id]["prompt_type"],
        "model_name": "Qwen/Qwen3-1.7B"
    })

# Save the reviewable dataset separately.
dataset_path = out_dir / "fatima_fellowship_dataset.json"

with dataset_path.open("w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

# Audit archive keeps both Case 1 outputs separately.
audit_path = out_dir / "fatima_fellowship_output_audit.json"

with audit_path.open("w", encoding="utf-8") as f:
    json.dump(recovered, f, ensure_ascii=False, indent=2)

print("Dataset created:", dataset_path.exists())
print("Dataset cases:", len(dataset))
print("Prompts attached:", sum(bool(r.get("prompt")) for r in dataset))
print("Records with raw output:", sum(bool(r.get("raw_output")) for r in dataset))
print("Separate output audit created:", audit_path.exists())
print("Case 1 outputs in audit:", sum(r["case_id"] == "case1" for r in recovered))


Dataset created: True
Dataset cases: 7
Prompts attached: 7
Records with raw output: 7
Separate output audit created: True
Case 1 outputs in audit: 2


In [22]:

import json
from pathlib import Path

path = Path("/content/fatima_fellowship_dataset.json")

with path.open("r", encoding="utf-8") as f:
    dataset = json.load(f)

required_fields = [
    "case_id",
    "topic",
    "reference_verdict",
    "model_verdict",
    "prompt",
    "raw_output",
    "model_name"
]

print("DATASET QUALITY CHECK")
print("=" * 45)

for record in dataset:
    missing = [
        field for field in required_fields
        if field not in record or record[field] is None
    ]

    print(
        record["case_id"],
        "| missing fields:", missing if missing else "None",
        "| prompt length:", len(record.get("prompt", "")),
        "| output length:", len(record.get("raw_output") or "")
    )

print("\nUnique case IDs:", len(set(r["case_id"] for r in dataset)))
print("Total records:", len(dataset))
print("All prompts non-empty:", all(r.get("prompt", "").strip() for r in dataset))
print("All outputs non-empty:", all(r.get("raw_output", "").strip() for r in dataset))
print("All records use expected model:", all(
    r.get("model_name") == "Qwen/Qwen3-1.7B" for r in dataset
))


DATASET QUALITY CHECK
case1 | missing fields: None | prompt length: 290 | output length: 415
case2 | missing fields: None | prompt length: 446 | output length: 539
case3 | missing fields: None | prompt length: 505 | output length: 653
case4 | missing fields: None | prompt length: 320 | output length: 214
case5 | missing fields: None | prompt length: 434 | output length: 252
case6 | missing fields: None | prompt length: 304 | output length: 322
case7 | missing fields: None | prompt length: 332 | output length: 278

Unique case IDs: 7
Total records: 7
All prompts non-empty: True
All outputs non-empty: True
All records use expected model: True


In [23]:

import json
from pathlib import Path
from collections import Counter

path = Path("/content/fatima_fellowship_dataset.json")

with path.open("r", encoding="utf-8") as f:
    data = json.load(f)

total = len(data)
correct_verdicts = sum(
    r["model_verdict"] == r["reference_verdict"]
    for r in data
)

reason_counts = Counter(
    r.get("reason_assessment", "NOT_RECORDED")
    for r in data
)

confidence_counts = Counter(
    r.get("verbal_confidence") or "NOT_PROVIDED"
    for r in data
)

print("PRELIMINARY RESULTS")
print("=" * 40)
print(f"Total cases: {total}")
print(
    f"Correct verdicts: {correct_verdicts}/{total} "
    f"({100 * correct_verdicts / total:.1f}%)"
)

print("\nReason assessment counts:")
for label, count in sorted(reason_counts.items()):
    print(f"{label}: {count}/{total}")

print("\nVerbal confidence labels:")
for label, count in sorted(confidence_counts.items()):
    print(f"{label}: {count}/{total}")

print("\nCase-level audit:")
for r in data:
    verdict_ok = r["model_verdict"] == r["reference_verdict"]
    print(
        r["case_id"],
        "| verdict correct:", verdict_ok,
        "| reason:", r.get("reason_assessment"),
        "| confidence:", r.get("verbal_confidence") or "NOT_PROVIDED"
    )

print("\nInterpretation:")
print("These are descriptive results from a small preliminary test set.")
print("Cases 3 and 4 use closely related proofs.")
print("Missing confidence labels are missing data, not low confidence.")


PRELIMINARY RESULTS
Total cases: 7
Correct verdicts: 6/7 (85.7%)

Reason assessment counts:
CORRECT: 4/7
INCORRECT: 1/7
PARTIALLY_INCORRECT: 2/7

Verbal confidence labels:
LOW: 1/7
MEDIUM: 2/7
NOT_PROVIDED: 4/7

Case-level audit:
case1 | verdict correct: True | reason: PARTIALLY_INCORRECT | confidence: MEDIUM
case2 | verdict correct: True | reason: CORRECT | confidence: LOW
case3 | verdict correct: False | reason: INCORRECT | confidence: MEDIUM
case4 | verdict correct: True | reason: CORRECT | confidence: NOT_PROVIDED
case5 | verdict correct: True | reason: CORRECT | confidence: NOT_PROVIDED
case6 | verdict correct: True | reason: PARTIALLY_INCORRECT | confidence: NOT_PROVIDED
case7 | verdict correct: True | reason: CORRECT | confidence: NOT_PROVIDED

Interpretation:
These are descriptive results from a small preliminary test set.
Cases 3 and 4 use closely related proofs.
Missing confidence labels are missing data, not low confidence.


In [24]:

import json
import torch
from pathlib import Path

assert "model" in globals() and model is not None, "Model is not loaded."
assert "tokenizer" in globals() and tokenizer is not None, "Tokenizer is not loaded."

additional_cases = [
    {
        "case_id": "case8",
        "topic": "Pointwise but not uniform on the real line",
        "reference_verdict": "INCORRECT",
        "reference_reason": "For every n, the supremum of |x/n| over R is infinite, so convergence is not uniform.",
        "prompt": """Let f_n(x)=x/n on the real line R, and let f(x)=0.
For every fixed real x, f_n(x) tends to zero.
A student concludes that the convergence is uniform on R.
Is the conclusion correct? Explain briefly.
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW"""
    },
    {
        "case_id": "case9",
        "topic": "A rational sequence on an unbounded interval",
        "reference_verdict": "INCORRECT",
        "reference_reason": "The supremum of x/(n+x) over x>=0 equals 1 for every n, so it does not tend to zero.",
        "prompt": """Let f_n(x)=x/(n+x) for x in [0,infinity), and let f(x)=0.
For every fixed x, f_n(x) tends to zero.
A student concludes that the convergence is uniform.
Is the conclusion correct? Explain briefly.
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW"""
    },
    {
        "case_id": "case10",
        "topic": "Sine sequence on a bounded interval",
        "reference_verdict": "CORRECT",
        "reference_reason": "Since |sin(x/n)|<=|x|/n<=1/n on [-1,1], the supremum error tends to zero.",
        "prompt": """Let f_n(x)=sin(x/n) on [-1,1], and let f(x)=0.
Since |sin(x/n)|<=|x|/n<=1/n for every x in [-1,1],
a student concludes that f_n converges uniformly to f.
Is the proof valid? Explain briefly.
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW"""
    },
    {
        "case_id": "case11",
        "topic": "Constant functions on an arbitrary nonempty domain",
        "reference_verdict": "CORRECT",
        "reference_reason": "The absolute error is exactly 1/n at every point, so its supremum is 1/n and tends to zero.",
        "prompt": """Let E be any nonempty set and define f_n(x)=1/n and f(x)=0 for all x in E.
A student claims that f_n converges uniformly to f on E because
sup over x in E of |f_n(x)-f(x)| equals 1/n, which tends to zero.
Is the claim correct? Explain briefly.
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW"""
    },
    {
        "case_id": "case12",
        "topic": "Uniform convergence using an algebraic bound",
        "reference_verdict": "CORRECT",
        "reference_reason": "For every real x, n+x^2>=2*sqrt(n)*|x|, so |x|/(n+x^2)<=1/(2*sqrt(n)), which tends to zero.",
        "prompt": """Let f_n(x)=x/(n+x^2) for all real x, and let f(x)=0.
Using n+x^2 >= 2*sqrt(n)*|x|, we obtain
|f_n(x)| <= 1/(2*sqrt(n)) for every real x.
A student concludes that f_n converges uniformly to zero on R.
Is the conclusion correct? Explain briefly.
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW"""
    },
    {
        "case_id": "case13",
        "topic": "Pointwise convergence with a nonvanishing supremum",
        "reference_verdict": "INCORRECT",
        "reference_reason": "For every n, the supremum of |n*x/(1+n^2*x^2)| over R is 1/2, so convergence is not uniform.",
        "prompt": """Let f_n(x)=n*x/(1+n^2*x^2) for all real x, and let f(x)=0.
For every fixed real x, f_n(x) tends to zero.
A student concludes that the convergence is uniform on R.
Is the conclusion correct? Explain briefly.
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW"""
    },
    {
        "case_id": "case14",
        "topic": "A boundary layer near zero",
        "reference_verdict": "INCORRECT",
        "reference_reason": "The pointwise limit is 1 at x=0 and 0 for x>0, but the supremum error on (0,1] equals 1 for every n.",
        "prompt": """Let f_n(x)=1/(1+n*x) on [0,1].
Its pointwise limit is f(0)=1 and f(x)=0 for every x>0.
A student claims that f_n converges uniformly to f on [0,1].
Is the claim correct? Explain briefly.
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW"""
    },
    {
        "case_id": "case15",
        "topic": "Uniform convergence on an unbounded interval",
        "reference_verdict": "CORRECT",
        "reference_reason": "For every x>=0, |x/(n*(1+x))|<=1/n, so the supremum error tends to zero.",
        "prompt": """Let f_n(x)=x/(n*(1+x)) for x in [0,infinity), and let f(x)=0.
For every x>=0, we have 0<=x/(1+x)<=1, so
|f_n(x)-f(x)|<=1/n.
A student concludes that f_n converges uniformly to zero.
Is the proof valid? Explain briefly.
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW"""
    }
]

new_results = []

for case in additional_cases:
    messages = [{"role": "user", "content": case["prompt"]}]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    record = {
        **case,
        "model_name": "Qwen/Qwen3-1.7B",
        "raw_output": raw_output,
        "raw_output_status": "RECORDED",
        "verdict_correct": None,
        "reason_assessment": "PENDING_MANUAL_REVIEW",
        "verbal_confidence": None
    }

    new_results.append(record)

    print("=" * 65)
    print("CASE:", case["case_id"], "-", case["topic"])
    print("REFERENCE VERDICT:", case["reference_verdict"])
    print("RAW MODEL OUTPUT:")
    print(raw_output)
    print()

output_path = Path("/content/fatima_fellowship_additional_cases.json")

with output_path.open("w", encoding="utf-8") as f:
    json.dump(new_results, f, ensure_ascii=False, indent=2)

print("New cases generated:", len(new_results))
print("Saved:", output_path)
print("File exists:", output_path.exists())
print("Manual review is still required.")


CASE: case8 - Pointwise but not uniform on the real line
REFERENCE VERDICT: INCORRECT
RAW MODEL OUTPUT:
Verdict: INCORRECT  
Reason: The convergence is not uniform because the uniform limit theorem requires that the convergence is uniform on a compact set, and R is not compact.

CASE: case9 - A rational sequence on an unbounded interval
REFERENCE VERDICT: INCORRECT
RAW MODEL OUTPUT:
Verdict: INCORRECT  
Reason: The convergence is not uniform because the maximum of |f_n(x) - f(x)| does not approach zero uniformly across all x in [0, ∞).

CASE: case10 - Sine sequence on a bounded interval
REFERENCE VERDICT: CORRECT
RAW MODEL OUTPUT:
Verdict: CORRECT  
Reason: The student correctly applies the uniform bound |sin(x/n)| ≤ |x|/n and shows that the bound tends to 0 as n → ∞, hence uniform convergence.

CASE: case11 - Constant functions on an arbitrary nonempty domain
REFERENCE VERDICT: CORRECT
RAW MODEL OUTPUT:
Verdict: CORRECT  
Reason: The student correctly notes that the supremum of |f_n(x

In [25]:

import json
from pathlib import Path

base = Path("/content")
source = base / "fatima_fellowship_additional_cases.json"
destination = base / "fatima_fellowship_additional_cases_reviewed.json"

with source.open("r", encoding="utf-8") as f:
    cases = json.load(f)

# Human review based on the mathematical reasoning in each raw answer.
review = {
    "case8": ("INCORRECT", None),
    "case9": ("PARTIALLY_INCORRECT", None),
    "case10": ("CORRECT", None),
    "case11": ("CORRECT", None),
    "case12": ("CORRECT", None),
    "case13": ("CORRECT", None),
    "case14": ("INCORRECT", None),
    "case15": ("CORRECT", None),
}

for case in cases:
    case_id = case["case_id"]
    reason_assessment, confidence = review[case_id]

    case["verdict_correct"] = (
        case["model_verdict"] == case["reference_verdict"]
        if case.get("model_verdict") else
        case["raw_output"].strip().startswith(
            "Verdict: " + case["reference_verdict"]
        )
    )

    case["reason_assessment"] = reason_assessment
    case["verbal_confidence"] = confidence
    case["review_status"] = "HUMAN_REVIEWED"
    case["review_note"] = {
        "case8": "Correct verdict, but compactness is not the reason; the supremum error is infinite.",
        "case9": "Correct verdict, but the explanation does not establish it with a mathematical bound.",
        "case10": "Correct verdict and valid uniform bound.",
        "case11": "Correct verdict and correct supremum calculation.",
        "case12": "Correct verdict and valid bound independent of x.",
        "case13": "Correct verdict and valid witness x=1/n gives error 1/2.",
        "case14": "Correct verdict, but the explanation cites an irrelevant singularity at x=-1.",
        "case15": "Correct verdict and valid uniform bound."
    }[case_id]

with destination.open("w", encoding="utf-8") as f:
    json.dump(cases, f, ensure_ascii=False, indent=2)

print("Reviewed file created:", destination.exists())
print("Cases reviewed:", len(cases))
print(
    "Correct verdicts:",
    sum(c["verdict_correct"] for c in cases),
    "/", len(cases)
)

from collections import Counter

print("\nReason assessment counts:")
for label, count in sorted(Counter(
    c["reason_assessment"] for c in cases
).items()):
    print(f"{label}: {count}")

print("\nConfidence provided:", sum(
    c["verbal_confidence"] is not None for c in cases
), "/", len(cases))

print("Original raw outputs preserved:", all(
    bool(c.get("raw_output")) for c in cases
))


Reviewed file created: True
Cases reviewed: 8
Correct verdicts: 8 / 8

Reason assessment counts:
CORRECT: 5
INCORRECT: 2
PARTIALLY_INCORRECT: 1

Confidence provided: 0 / 8
Original raw outputs preserved: True


In [26]:

import json
import re
from pathlib import Path
from collections import Counter

base = Path("/content")

with open(base / "fatima_fellowship_dataset.json", "r", encoding="utf-8") as f:
    original_cases = json.load(f)

with open(base / "fatima_fellowship_additional_cases_reviewed.json", "r", encoding="utf-8") as f:
    additional_cases = json.load(f)

combined = []

# Keep the original seven cases and their existing assessments.
for case in original_cases:
    item = dict(case)
    item["dataset_group"] = "initial"
    combined.append(item)

# Add the eight reviewed cases.
for case in additional_cases:
    item = dict(case)

    # Extract the model's stated verdict from the raw output.
    match = re.search(
        r"Verdict:\s*(CORRECT|INCORRECT)",
        item.get("raw_output", ""),
        flags=re.IGNORECASE
    )

    if match:
        item["model_verdict"] = match.group(1).upper()

    item["verdict_correct"] = (
        item.get("model_verdict") == item.get("reference_verdict")
    )
    item["dataset_group"] = "additional"
    combined.append(item)

# Save the combined dataset separately.
combined_path = base / "fatima_fellowship_combined_dataset.json"

with combined_path.open("w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

total = len(combined)
correct_verdicts = sum(
    c.get("verdict_correct") is True for c in combined
)

reason_counts = Counter(
    c.get("reason_assessment", "NOT_RECORDED")
    for c in combined
)

confidence_counts = Counter(
    c.get("verbal_confidence") or "NOT_PROVIDED"
    for c in combined
)

print("COMBINED DATASET")
print("=" * 40)
print("File created:", combined_path.exists())
print("Total cases:", total)
print("Unique case IDs:", len(set(c["case_id"] for c in combined)))
print(f"Correct verdicts: {correct_verdicts}/{total} ({100*correct_verdicts/total:.1f}%)")

print("\nReason assessment counts:")
for label, count in sorted(reason_counts.items()):
    print(f"{label}: {count}/{total}")

print("\nConfidence counts:")
for label, count in sorted(confidence_counts.items()):
    print(f"{label}: {count}/{total}")

print("\nCases with missing raw outputs:", sum(
    not c.get("raw_output") for c in combined
))
print("All verdicts parsed:", all(
    c.get("model_verdict") in ["CORRECT", "INCORRECT"]
    for c in combined
))


COMBINED DATASET
File created: True
Total cases: 15
Unique case IDs: 15
Correct verdicts: 14/15 (93.3%)

Reason assessment counts:
CORRECT: 9/15
INCORRECT: 3/15
PARTIALLY_INCORRECT: 3/15

Confidence counts:
LOW: 1/15
MEDIUM: 2/15
NOT_PROVIDED: 12/15

Cases with missing raw outputs: 0
All verdicts parsed: True


In [27]:

import json
from pathlib import Path

path = Path("/content/fatima_fellowship_combined_dataset.json")

with path.open("r", encoding="utf-8") as f:
    cases = json.load(f)

print(f"CASE-LEVEL AUDIT — {len(cases)} cases")
print("=" * 90)

for case in cases:
    print(f"\nCase ID: {case.get('case_id', 'MISSING_ID')}")
    print("Group:", case.get("dataset_group", "NOT_RECORDED"))
    print("Reference verdict:", case.get("reference_verdict", "MISSING"))
    print("Model verdict:", case.get("model_verdict", "MISSING"))
    print("Verdict correct:", case.get("verdict_correct", "MISSING"))
    print("Reason assessment:", case.get("reason_assessment", "NOT_RECORDED"))
    print("Verbal confidence:", case.get("verbal_confidence", "NOT_PROVIDED"))

    raw = case.get("raw_output", "")
    print("Raw output preview:", raw[:350].replace("\n", " "))

print("\n" + "=" * 90)
print("Audit finished. Original combined file was not modified.")


CASE-LEVEL AUDIT — 15 cases

Case ID: case1
Group: initial
Reference verdict: INCORRECT
Model verdict: INCORRECT
Verdict correct: True
Reason assessment: PARTIALLY_INCORRECT
Verbal confidence: MEDIUM
Raw output preview: Verdict: INCORRECT   Reason: Pointwise convergence does not imply uniform convergence because, for example, the sequence $ f_n(x) = x^n $ converges pointwise to 0 on [0,1], but does not converge uniformly.   Confidence: MEDIUM   Why this confidence: The uniform convergence would require the maximum difference between $ f_n(x) $ and 0 to approach 0 

Case ID: case2
Group: initial
Reference verdict: INCORRECT
Model verdict: INCORRECT
Verdict correct: True
Reason assessment: CORRECT
Verbal confidence: LOW
Raw output preview: Verdict: INCORRECT   Reason: The proof incorrectly states that a sequence of functions converges uniformly to f on E because for every x in E, there exists N depending on x such that for all n ≥ N, |f_n(x) - f(x)| < ε. The correct definition of uniform

In [28]:

import json
import pandas as pd
from pathlib import Path

base = Path("/content")

with open(
    base / "fatima_fellowship_combined_dataset.json",
    "r",
    encoding="utf-8"
) as f:
    cases = json.load(f)

# Human-reviewed notes for cases with identified reasoning issues.
error_notes = {
    "case1": (
        "Incorrect pointwise-limit statement at x=1",
        "Boundary / limit identification"
    ),
    "case3": (
        "Rejected a valid uniform-convergence proof",
        "False mathematical objection"
    ),
    "case6": (
        "Confused maximum with supremum on [0,1)",
        "Boundary / supremum handling"
    ),
    "case8": (
        "Incorrectly claimed compactness is required",
        "False theorem or condition"
    ),
    "case9": (
        "Correct conclusion without sufficient justification",
        "Insufficient mathematical justification"
    ),
    "case14": (
        "Used an irrelevant singularity outside the domain",
        "Irrelevant mathematical argument"
    ),
}

rows = []

for c in cases:
    case_id = c.get("case_id", "unknown")
    note, category = error_notes.get(
        case_id,
        ("No specific issue flagged in this review", "No flagged issue")
    )

    rows.append({
        "case_id": case_id,
        "dataset_group": c.get("dataset_group"),
        "reference_verdict": c.get("reference_verdict"),
        "model_verdict": c.get("model_verdict"),
        "verdict_correct": c.get("verdict_correct"),
        "reason_assessment": c.get("reason_assessment"),
        "verbal_confidence": c.get("verbal_confidence"),
        "error_category": category,
        "human_review_note": note,
        "raw_output": c.get("raw_output", "")
    })

df = pd.DataFrame(rows)

csv_path = base / "fatima_fellowship_error_analysis.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Error analysis CSV created:", csv_path.exists())
print("Rows:", len(df))

print("\nVerdict correctness:")
print(df["verdict_correct"].value_counts(dropna=False))

print("\nReason assessment:")
print(df["reason_assessment"].value_counts(dropna=False))

print("\nCases with specific reasoning issues:")
print(df[df["error_category"] != "No flagged issue"][
    ["case_id", "reason_assessment", "error_category"]
].to_string(index=False))

print("\nFile:", csv_path)


Error analysis CSV created: True
Rows: 15

Verdict correctness:
verdict_correct
True     14
False     1
Name: count, dtype: int64

Reason assessment:
reason_assessment
CORRECT                9
PARTIALLY_INCORRECT    3
INCORRECT              3
Name: count, dtype: int64

Cases with specific reasoning issues:
case_id   reason_assessment                          error_category
  case1 PARTIALLY_INCORRECT         Boundary / limit identification
  case3           INCORRECT            False mathematical objection
  case6 PARTIALLY_INCORRECT            Boundary / supremum handling
  case8           INCORRECT              False theorem or condition
  case9 PARTIALLY_INCORRECT Insufficient mathematical justification
 case14           INCORRECT        Irrelevant mathematical argument

File: /content/fatima_fellowship_error_analysis.csv


In [29]:

import json
import torch
from pathlib import Path

new_cases = [
    {
        "case_id": "case16",
        "topic": "uniform_bound",
        "prompt": """
Determine whether the following claim and proof are correct.

Claim: f_n(x) = x^n/n converges uniformly to 0 on [0,1].

Proof: For every x in [0,1], |f_n(x)| = x^n/n <= 1/n.
Since 1/n tends to 0 independently of x, the convergence is uniform.

Evaluate the proof, not just the claim.
""",
        "reference_verdict": "CORRECT",
        "reference_reason": "The uniform error is bounded by 1/n, which tends to zero."
    },
    {
        "case_id": "case17",
        "topic": "unbounded_domain",
        "prompt": """
Determine whether the following claim and proof are correct.

Claim: f_n(x) = (x/(1+x))^n converges uniformly to 0 on [0,infinity).

Proof: For every fixed x >= 0, we have 0 <= x/(1+x) < 1,
so f_n(x) tends to 0. Therefore the convergence is uniform.

Evaluate the proof, not just the claim.
""",
        "reference_verdict": "INCORRECT",
        "reference_reason": "Pointwise convergence does not imply uniform convergence. The supremum of (x/(1+x))^n over x >= 0 is 1 for every n."
    },
    {
        "case_id": "case18",
        "topic": "uniform_bound",
        "prompt": """
Determine whether the following claim and proof are correct.

Claim: f_n(x) = sin(nx)/n converges uniformly to 0 on the real line.

Proof: For every real x, |sin(nx)/n| <= 1/n.
The bound tends to zero and does not depend on x, so the convergence is uniform.

Evaluate the proof, not just the claim.
""",
        "reference_verdict": "CORRECT",
        "reference_reason": "The absolute error is bounded by 1/n for every real x."
    },
    {
        "case_id": "case19",
        "topic": "pointwise_limit_and_uniformity",
        "prompt": """
Determine whether the following claim and proof are correct.

Let f_n(x) = exp(-n*x) on [0,1], and define
f(0)=1 and f(x)=0 for every x in (0,1].

Claim: f_n converges uniformly to f on [0,1].

Proof: For each fixed x > 0, exp(-n*x) tends to zero, and at x=0
the sequence equals 1. Thus it converges pointwise to f, so it
converges uniformly.

Evaluate the proof, not just the pointwise-limit statement.
""",
        "reference_verdict": "INCORRECT",
        "reference_reason": "For x>0 the error is exp(-n*x), whose supremum over (0,1] is 1 for every n; it does not tend to zero."
    },
    {
        "case_id": "case20",
        "topic": "uniform_bound",
        "prompt": """
Determine whether the following claim and proof are correct.

Claim: f_n(x) = x/(1+n*x) converges uniformly to 0 on [0,1].

Proof: For every x in [0,1], 0 <= x/(1+n*x) <= 1/n.
Since 1/n tends to zero, the convergence is uniform.

Evaluate the proof, not just the claim.
""",
        "reference_verdict": "CORRECT",
        "reference_reason": "The error is bounded above by 1/n uniformly on [0,1]."
    },
    {
        "case_id": "case21",
        "topic": "compactness_and_continuity",
        "prompt": """
Determine whether the following mathematical statement is correct.

Statement: If continuous functions f_n on [0,1] converge pointwise
to a continuous function f, then f_n necessarily converge uniformly
to f, because [0,1] is compact.

Give a verdict and justify it mathematically.
""",
        "reference_verdict": "INCORRECT",
        "reference_reason": "Compactness and continuity of every function and the limit do not by themselves imply uniform convergence. Additional conditions, such as those in Dini's theorem, are needed."
    },
    {
        "case_id": "case22",
        "topic": "uniform_bound",
        "prompt": """
Determine whether the following claim and proof are correct.

Claim: f_n(x) = 1/(n+x) converges uniformly to 0 on [0,infinity).

Proof: For all x >= 0, 0 < 1/(n+x) <= 1/n.
Since 1/n tends to zero, the convergence is uniform.

Evaluate the proof, not just the claim.
""",
        "reference_verdict": "CORRECT",
        "reference_reason": "The error is bounded by 1/n uniformly over the entire domain."
    },
    {
        "case_id": "case23",
        "topic": "unbounded_domain",
        "prompt": """
Determine whether the following claim and proof are correct.

Claim: f_n(x) = x/n converges uniformly to 0 on [0,infinity).

Proof: For every fixed x >= 0, x/n tends to zero as n tends to
infinity. Therefore the convergence is uniform.

Evaluate the proof, not just the claim.
""",
        "reference_verdict": "INCORRECT",
        "reference_reason": "The convergence is pointwise but not uniform; the error is unbounded over the domain for every fixed n."
    },
    {
        "case_id": "case24",
        "topic": "uniform_bound",
        "prompt": """
Determine whether the following claim and proof are correct.

Claim: f_n(x) = x^2/(n+x^2) converges uniformly to 0 on [0,1].

Proof: For every x in [0,1],
0 <= x^2/(n+x^2) <= 1/n.
Since 1/n tends to zero independently of x, the convergence is uniform.

Evaluate the proof, not just the claim.
""",
        "reference_verdict": "CORRECT",
        "reference_reason": "Since x^2 <= 1 and n+x^2 >= n, the error is at most 1/n."
    },
    {
        "case_id": "case25",
        "topic": "definition_of_uniform_convergence",
        "prompt": """
Determine whether the following proof establishes uniform convergence
of f_n to f on a set E.

Proof: For every epsilon > 0 and every x in E, there exists an
integer N depending on epsilon and x such that for all n >= N,
|f_n(x)-f(x)| < epsilon. Therefore f_n converges uniformly to f.

Evaluate whether the quantifiers in this proof are sufficient.
""",
        "reference_verdict": "INCORRECT",
        "reference_reason": "The integer N is allowed to depend on x. Uniform convergence requires one N for each epsilon that works for every x in E."
    }
]

def get_model_text(prompt):
    messages = [
        {
            "role": "system",
            "content": (
                "You are evaluating mathematical claims about uniform convergence. "
                "Be mathematically precise. Do not assume a proof is correct merely "
                "because its conclusion is true. Follow the requested output format."
            )
        },
        {
            "role": "user",
            "content": prompt + """

Return exactly these sections:
Verdict: CORRECT or INCORRECT
Reason: Explain the mathematical justification, including a relevant bound,
supremum, counterexample, or quantifier issue when appropriate.
Confidence: LOW, MEDIUM, or HIGH
Why this confidence: Briefly explain your confidence in your verdict.
"""
        }
    ]

    # Use the tokenizer's chat template when available.
    if getattr(tokenizer, "chat_template", None):
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        input_text = (
            messages[0]["content"] + "\n\n" + messages[1]["content"]
        )

    inputs = tokenizer(input_text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


results = []

for i, case in enumerate(new_cases, start=1):
    print(f"Running {case['case_id']} ({i}/{len(new_cases)})...")

    try:
        raw_output = get_model_text(case["prompt"])
        status = "SUCCESS"
    except Exception as e:
        raw_output = ""
        status = f"ERROR: {type(e).__name__}: {e}"

    results.append({
        "case_id": case["case_id"],
        "topic": case["topic"],
        "prompt": case["prompt"],
        "reference_verdict": case["reference_verdict"],
        "reference_reason": case["reference_reason"],
        "raw_output": raw_output,
        "run_status": status
    })

    print(raw_output[:500] if raw_output else status)
    print("-" * 60)

output_path = Path("/content/fatima_fellowship_cases16_25.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\nNew test set saved:", output_path.exists())
print("Cases saved:", len(results))
print("Successful runs:", sum(r["run_status"] == "SUCCESS" for r in results))
print("File:", output_path)


Running case16 (1/10)...
<think>
Okay, let's tackle this problem. The claim is that f_n(x) = x^n/n converges uniformly to 0 on [0,1]. The proof given says that for every x in [0,1], |f_n(x)| = x^n/n ≤ 1/n, and since 1/n tends to 0 independently of x, the convergence is uniform.

First, I need to recall what uniform convergence means. Uniform convergence of a sequence of functions {f_n} to a function f on a set S means that for every ε > 0, there exists an N such that for all n ≥ N and all x in S, |f_n(x) - f(x)| < ε. In
------------------------------------------------------------
Running case17 (2/10)...
<think>
Okay, let's tackle this problem. The claim is that the sequence of functions f_n(x) = (x/(1+x))^n converges uniformly to 0 on [0, infinity). The proof given says that for each fixed x ≥ 0, 0 ≤ x/(1+x) < 1, so f_n(x) tends to 0, hence the convergence is uniform.

First, I need to recall what uniform convergence means. Uniform convergence requires that for every ε > 0, there exis

In [30]:

import json
from pathlib import Path

path = Path("/content/fatima_fellowship_cases16_25.json")

with path.open("r", encoding="utf-8") as f:
    results = json.load(f)

print("RESPONSE COMPLETENESS AUDIT")
print("=" * 65)

for case in results:
    output = case.get("raw_output", "")

    has_verdict = "Verdict:" in output
    has_confidence = "Confidence:" in output
    has_reason = "Reason:" in output

    print(f"\n{case['case_id']}")
    print("Characters:", len(output))
    print("Has verdict:", has_verdict)
    print("Has reason:", has_reason)
    print("Has confidence:", has_confidence)
    print("Last 350 characters:")
    print(output[-350:] if output else "[EMPTY OUTPUT]")
    print("-" * 65)

print("\nTotal cases:", len(results))
print("Cases containing verdict:", sum(
    "Verdict:" in c.get("raw_output", "") for c in results
))
print("Cases containing confidence:", sum(
    "Confidence:" in c.get("raw_output", "") for c in results
))


RESPONSE COMPLETENESS AUDIT

case16
Characters: 788
Has verdict: False
Has reason: False
Has confidence: False
Last 350 characters:
uch that for all n ≥ N and all x in S, |f_n(x) - f(x)| < ε. In this case, f(x) is 0, so the condition becomes |f_n(x)| < ε for all x in [0,1] and n ≥ N.

The proof claims that |f_n(x)| ≤ 1/n, which is true because x^n ≤ 1 for all x in [0,1], so x^n/n ≤ 1/n. But wait, the problem is that the bound 1/n is dependent on n, not on x. However, in uniform
-----------------------------------------------------------------

case17
Characters: 901
Has verdict: False
Has reason: False
Has confidence: False
Last 350 characters:
nce requires a bound on the maximum of |f_n(x)| over the domain, not just pointwise convergence.

The proof mentions that 0 ≤ x/(1+x) < 1, which is true, but that doesn't directly address the uniform convergence. The key here is to find the supremum of |f_n(x)| on [0, ∞) and see if it can be made arbitrarily small as n increases.

Let's analyze 

In [31]:

import json
import torch
from pathlib import Path

base = Path("/content")
source_path = base / "fatima_fellowship_cases16_25.json"

with source_path.open("r", encoding="utf-8") as f:
    old_results = json.load(f)

def generate_final_answer(prompt):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a mathematics evaluator. "
                "Return only the requested final answer. "
                "Do not include analysis, scratch work, or a think section."
            )
        },
        {
            "role": "user",
            "content": prompt + """

Use exactly this concise format:
Verdict: CORRECT or INCORRECT
Reason: One or two sentences with the decisive mathematical argument.
Confidence: LOW, MEDIUM, or HIGH
Why this confidence: One short sentence.
"""
        }
    ]

    # Disable Qwen3 thinking mode in the chat template.
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(input_text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


rerun_results = []

for i, case in enumerate(old_results, start=1):
    print(f"Re-running {case['case_id']} ({i}/{len(old_results)})...")

    try:
        answer = generate_final_answer(case["prompt"])
        status = "SUCCESS" if "Verdict:" in answer else "CHECK_OUTPUT"
    except Exception as e:
        answer = ""
        status = f"ERROR: {type(e).__name__}: {e}"

    rerun_results.append({
        **case,
        "raw_output_previous_run": case.get("raw_output", ""),
        "raw_output": answer,
        "run_status": status
    })

    print(answer[:450] if answer else status)
    print("-" * 55)

output_path = base / "fatima_fellowship_cases16_25_rerun.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(rerun_results, f, ensure_ascii=False, indent=2)

print("\nRERUN SUMMARY")
print("File created:", output_path.exists())
print("Cases:", len(rerun_results))
print("Outputs containing verdict:", sum(
    "Verdict:" in c["raw_output"] for c in rerun_results
))
print("Outputs containing confidence:", sum(
    "Confidence:" in c["raw_output"] for c in rerun_results
))
print("Run status counts:")
for status in sorted(set(c["run_status"] for c in rerun_results)):
    print(status, sum(c["run_status"] == status for c in rerun_results))

print("Saved to:", output_path)


Re-running case16 (1/10)...
Verdict: INCORRECT  
Reason: The proof is incorrect because the bound $ \frac{1}{n} $ does not hold for all $ x \in [0,1] $. For $ x = 1 $, $ |f_n(1)| = \frac{1}{n} $, but for $ x = 0 $, $ |f_n(0)| = 0 $. The bound $ \frac{1}{n} $ is not valid for all $ x \in [0,1] $, so the uniform convergence is not established.  
Confidence: LOW  
Why this confidence: The bound $ \frac{1}{n} $ is not valid for all $ x \in [0,1] $, so the proof fails to establi
-------------------------------------------------------
Re-running case17 (2/10)...
Verdict: INCORRECT  
Reason: The proof is incorrect because it assumes pointwise convergence to 0 but fails to show uniform convergence. The function $ \frac{x}{1+x} $ is less than 1 for all $ x \geq 0 $, but the maximum value of $ \frac{x}{1+x} $ on $[0, \infty)$ is 1 (achieved as $ x \to \infty $), so the sequence $ f_n(x) $ does not converge uniformly to 0 on $[0, \infty)$.

Confidence: LOW  
Why this confidence: The proof incorre

In [32]:

import json
import re
from pathlib import Path

path = Path("/content/fatima_fellowship_cases16_25_rerun.json")

with path.open("r", encoding="utf-8") as f:
    cases = json.load(f)

print("HUMAN REVIEW — CASES 16–25")
print("=" * 75)

for c in cases:
    output = c.get("raw_output", "")

    verdict = re.search(
        r"Verdict:\s*(CORRECT|INCORRECT)",
        output,
        re.IGNORECASE
    )
    confidence = re.search(
        r"Confidence:\s*(LOW|MEDIUM|HIGH)",
        output,
        re.IGNORECASE
    )
    reason = re.search(
        r"Reason:\s*(.*?)(?=\n\s*Confidence:|\Z)",
        output,
        re.IGNORECASE | re.DOTALL
    )

    model_verdict = verdict.group(1).upper() if verdict else "MISSING"
    model_confidence = confidence.group(1).upper() if confidence else "MISSING"

    print(f"\n{c['case_id']} | Topic: {c.get('topic')}")
    print("Reference:", c.get("reference_verdict"))
    print("Model:", model_verdict)
    print("Confidence:", model_confidence)
    print("Reason:", reason.group(1).strip() if reason else "MISSING")
    print("Output characters:", len(output))
    print("-" * 75)

print("\nConfidence availability:",
      sum(bool(re.search(r"Confidence:\s*(LOW|MEDIUM|HIGH)",
                         c.get("raw_output", ""), re.IGNORECASE))
          for c in cases),
      "/", len(cases))


HUMAN REVIEW — CASES 16–25

case16 | Topic: uniform_bound
Reference: CORRECT
Model: INCORRECT
Confidence: LOW
Reason: The proof is incorrect because the bound $ \frac{1}{n} $ does not hold for all $ x \in [0,1] $. For $ x = 1 $, $ |f_n(1)| = \frac{1}{n} $, but for $ x = 0 $, $ |f_n(0)| = 0 $. The bound $ \frac{1}{n} $ is not valid for all $ x \in [0,1] $, so the uniform convergence is not established.
Output characters: 473
---------------------------------------------------------------------------

case17 | Topic: unbounded_domain
Reference: INCORRECT
Model: INCORRECT
Confidence: LOW
Reason: The proof is incorrect because it assumes pointwise convergence to 0 but fails to show uniform convergence. The function $ \frac{x}{1+x} $ is less than 1 for all $ x \geq 0 $, but the maximum value of $ \frac{x}{1+x} $ on $[0, \infty)$ is 1 (achieved as $ x \to \infty $), so the sequence $ f_n(x) $ does not converge uniformly to 0 on $[0, \infty)$.
Output characters: 515
--------------------------

In [33]:

import json
from pathlib import Path
from collections import Counter

base = Path("/content")

with open(base / "fatima_fellowship_cases16_25_rerun.json",
          "r", encoding="utf-8") as f:
    new_cases = json.load(f)

with open(base / "fatima_fellowship_combined_dataset.json",
          "r", encoding="utf-8") as f:
    previous_cases = json.load(f)

# Human review of each new model response.
review = {
    "case16": {
        "reason_assessment": "INCORRECT",
        "note": "Rejected a valid uniform bound: 0 <= x^n/n <= 1/n."
    },
    "case17": {
        "reason_assessment": "PARTIALLY_INCORRECT",
        "note": "Correct verdict and main idea, but calls 1 a maximum; it is a supremum not attained."
    },
    "case18": {
        "reason_assessment": "INCORRECT",
        "note": "The valid uniform bound 1/n tends to zero, so the model's objection is false."
    },
    "case19": {
        "reason_assessment": "PARTIALLY_INCORRECT",
        "note": "Correct verdict, but the explanation contradicts itself about the pointwise limit. Nonuniformity follows because the supremum error is 1 for every n."
    },
    "case20": {
        "reason_assessment": "INCORRECT",
        "note": "The inequality x/(1+n*x) <= 1/n is valid for every x >= 0."
    },
    "case21": {
        "reason_assessment": "PARTIALLY_INCORRECT",
        "note": "Verdict is correct, but x^n on [0,1] has a discontinuous pointwise limit, so it does not refute the exact statement."
    },
    "case22": {
        "reason_assessment": "INCORRECT",
        "note": "The inequality 1/(n+x) <= 1/n is valid for x >= 0; it proves uniform convergence."
    },
    "case23": {
        "reason_assessment": "PARTIALLY_INCORRECT",
        "note": "Correct verdict, but the explanation lacks a decisive witness or explicit supremum argument."
    },
    "case24": {
        "reason_assessment": "INCORRECT",
        "note": "The valid bound x^2/(n+x^2) <= 1/n proves uniform convergence on [0,1]."
    },
    "case25": {
        "reason_assessment": "INCORRECT",
        "note": "Incorrectly accepts an argument in which N may depend on x; uniform convergence requires N independent of x."
    }
}

reviewed_new = []

for case in new_cases:
    item = dict(case)
    case_id = item["case_id"]
    item.update(review[case_id])

    item["model_verdict"] = (
        "CORRECT" if "Verdict: CORRECT" in item.get("raw_output", "")
        else "INCORRECT" if "Verdict: INCORRECT" in item.get("raw_output", "")
        else "MISSING"
    )

    item["verbal_confidence"] = next(
        (level for level in ["LOW", "MEDIUM", "HIGH"]
         if f"Confidence: {level}" in item.get("raw_output", "")),
        None
    )

    item["verdict_correct"] = (
        item["model_verdict"] == item["reference_verdict"]
    )
    item["human_review_status"] = "REVIEWED"

    # Correct the flawed explanation in the original reference for Case 21.
    if case_id == "case21":
        item["reference_reason"] = (
            "The statement is false. A counterexample is "
            "f_n(x) = max(0, 1 - n^2*abs(x - 1/n)) on [0,1]. "
            "Each f_n is continuous and f_n(x) tends pointwise to the "
            "continuous zero function, but sup_x |f_n(x)| = 1 for every n."
        )
        item["reference_reason_corrected_by_human_review"] = True

    reviewed_new.append(item)

reviewed_path = base / "fatima_fellowship_cases16_25_reviewed.json"
with reviewed_path.open("w", encoding="utf-8") as f:
    json.dump(reviewed_new, f, ensure_ascii=False, indent=2)

# Combine with the original 15 cases without modifying their source file.
combined_25 = [dict(c) for c in previous_cases] + reviewed_new
combined_path = base / "fatima_fellowship_combined_25_reviewed.json"

with combined_path.open("w", encoding="utf-8") as f:
    json.dump(combined_25, f, ensure_ascii=False, indent=2)

print("Reviewed new file created:", reviewed_path.exists())
print("Combined file created:", combined_path.exists())
print("Total cases:", len(combined_25))
print("Unique case IDs:", len(set(c["case_id"] for c in combined_25)))

print("\nVerdict accuracy:")
correct = sum(c.get("verdict_correct") is True for c in combined_25)
print(f"{correct}/{len(combined_25)} = {100*correct/len(combined_25):.1f}%")

print("\nReason assessments:")
for label, count in sorted(Counter(
    c.get("reason_assessment", "NOT_RECORDED")
    for c in combined_25
).items()):
    print(f"{label}: {count}")

print("\nConfidence labels:")
for label, count in sorted(Counter(
    c.get("verbal_confidence") or "NOT_PROVIDED"
    for c in combined_25
).items()):
    print(f"{label}: {count}")

print("\nNew cases with successful human review:", len(reviewed_new))
print("Case 21 reference corrected: True")


Reviewed new file created: True
Combined file created: True
Total cases: 25
Unique case IDs: 25

Verdict accuracy:
18/25 = 72.0%

Reason assessments:
CORRECT: 9
INCORRECT: 9
PARTIALLY_INCORRECT: 7

Confidence labels:
HIGH: 1
LOW: 7
MEDIUM: 5
NOT_PROVIDED: 12

New cases with successful human review: 10
Case 21 reference corrected: True


In [34]:

import json
import pandas as pd
from pathlib import Path

path = Path("/content/fatima_fellowship_combined_25_reviewed.json")

with path.open("r", encoding="utf-8") as f:
    cases = json.load(f)

df = pd.DataFrame(cases)

df["confidence_group"] = (
    df["verbal_confidence"]
    .fillna("NOT_PROVIDED")
    .str.upper()
)

df["verdict_correct"] = df["verdict_correct"].astype(bool)

# Count correct and incorrect verdicts at each confidence level.
table = pd.crosstab(
    df["confidence_group"],
    df["verdict_correct"]
).reindex(
    ["LOW", "MEDIUM", "HIGH", "NOT_PROVIDED"],
    fill_value=0
)

table = table.rename(columns={
    False: "Wrong verdict",
    True: "Correct verdict"
})

table["Total"] = table.sum(axis=1)
table["Accuracy (%)"] = (
    100 * table["Correct verdict"] / table["Total"]
).round(1)

print("VERBAL CONFIDENCE VS VERDICT CORRECTNESS")
print("=" * 60)
print(table.to_string())

print("\nConfidence coverage:")
provided = (df["confidence_group"] != "NOT_PROVIDED").sum()
print(f"{provided}/{len(df)} ({100*provided/len(df):.1f}%)")

print("\nCases with HIGH confidence:")
high_cases = df[df["confidence_group"] == "HIGH"]
if len(high_cases):
    print(high_cases[[
        "case_id", "reference_verdict", "model_verdict",
        "verdict_correct", "reason_assessment"
    ]].to_string(index=False))
else:
    print("No HIGH-confidence cases.")

print("\nInterpretation note:")
print(
    "These are exploratory results from a small, topic-specific dataset. "
    "Verbal confidence categories are not calibrated probabilities."
)


VERBAL CONFIDENCE VS VERDICT CORRECTNESS
verdict_correct   Wrong verdict  Correct verdict  Total  Accuracy (%)
confidence_group                                                     
LOW                           2                5      7          71.4
MEDIUM                        4                1      5          20.0
HIGH                          1                0      1           0.0
NOT_PROVIDED                  0               12     12         100.0

Confidence coverage:
13/25 (52.0%)

Cases with HIGH confidence:
case_id reference_verdict model_verdict  verdict_correct reason_assessment
 case25         INCORRECT       CORRECT            False         INCORRECT

Interpretation note:
These are exploratory results from a small, topic-specific dataset. Verbal confidence categories are not calibrated probabilities.


In [35]:

import os
import json

base = "/content"

print("=== Relevant project files ===")
for name in sorted(os.listdir(base)):
    if "fatima_fellowship" in name.lower():
        path = os.path.join(base, name)
        if os.path.isfile(path):
            print(f"{name} ({os.path.getsize(path):,} bytes)")

print("\n=== Saved generation settings, if available ===")
for name in sorted(os.listdir(base)):
    if "config" in name.lower() and "fatima" in name.lower():
        path = os.path.join(base, name)
        if os.path.isfile(path):
            print(f"\n--- {name} ---")
            try:
                with open(path, "r", encoding="utf-8") as f:
                    print(f.read()[:3000])
            except Exception as e:
                print("Could not read:", e)


=== Relevant project files ===
fatima_fellowship_additional_cases.json (7,718 bytes)
fatima_fellowship_additional_cases_reviewed.json (8,597 bytes)
fatima_fellowship_cases16_25.json (15,290 bytes)
fatima_fellowship_cases16_25_rerun.json (20,304 bytes)
fatima_fellowship_cases16_25_reviewed.json (23,216 bytes)
fatima_fellowship_combined_25_reviewed.json (41,787 bytes)
fatima_fellowship_combined_dataset.json (18,573 bytes)
fatima_fellowship_dataset.json (9,210 bytes)
fatima_fellowship_error_analysis.csv (6,064 bytes)
fatima_fellowship_output_audit.json (3,590 bytes)
fatima_fellowship_recovered_outputs.json (3,590 bytes)
fatima_fellowship_results.csv (4,329 bytes)
fatima_fellowship_results.json (5,872 bytes)

=== Saved generation settings, if available ===


In [37]:

import json
import re
import pandas as pd

path = "/content/fatima_fellowship_combined_25_reviewed.json"

with open(path, "r", encoding="utf-8") as f:
    cases = json.load(f)

rows = []

for c in cases:
    output = c.get("raw_output") or ""
    prompt = c.get("prompt") or ""

    confidence_match = re.search(
        r"Confidence:\s*(HIGH|MEDIUM|LOW)\b",
        output,
        re.IGNORECASE
    )
    verdict_match = re.search(
        r"Verdict:\s*(CORRECT|INCORRECT)\b",
        output,
        re.IGNORECASE
    )
    reason_match = re.search(
        r"Reason:\s*",
        output,
        re.IGNORECASE
    )

    rows.append({
        "case_id": c.get("case_id"),
        "prompt_type": c.get("prompt_type", "not_recorded"),
        "confidence_requested": bool(
            re.search(r"confidence", prompt, re.IGNORECASE)
        ),
        "confidence_in_raw_output": (
            confidence_match.group(1).upper()
            if confidence_match else "MISSING"
        ),
        "verdict_found": bool(verdict_match),
        "reason_found": bool(reason_match),
        "raw_output_status": c.get("raw_output_status", "not_recorded")
    })

audit = pd.DataFrame(rows)

print(audit.to_string(index=False))
print("\n=== Summary ===")
print("Cases:", len(audit))
print("Confidence requested:", audit["confidence_requested"].sum())
print("Confidence present:", (audit["confidence_in_raw_output"] != "MISSING").sum())
print("Verdict found:", audit["verdict_found"].sum())
print("Reason found:", audit["reason_found"].sum())


case_id    prompt_type  confidence_requested confidence_in_raw_output  verdict_found  reason_found raw_output_status
  case1   short_format                  True                   MEDIUM           True          True          RECORDED
  case2       original                  True                      LOW           True          True          RECORDED
  case3       original                  True                   MEDIUM           True          True          RECORDED
  case4     paraphrase                  True                  MISSING           True          True          RECORDED
  case5 candidate_case                  True                  MISSING           True          True          RECORDED
  case6 candidate_case                  True                  MISSING           True          True          RECORDED
  case7 candidate_case                  True                  MISSING           True          True          RECORDED
  case8   not_recorded                  True                  MI

In [38]:

import torch
import transformers

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nNow check the earlier notebook cells where you:")
print("1. Loaded Qwen/Qwen3-1.7B")
print("2. Created the tokenizer and model")
print("3. Called model.generate()")
print("4. Set enable_thinking and max_new_tokens")


PyTorch version: 2.11.0+cu130
Transformers version: 5.20.1
CUDA available: True
GPU: Tesla T4

Now check the earlier notebook cells where you:
1. Loaded Qwen/Qwen3-1.7B
2. Created the tokenizer and model
3. Called model.generate()
4. Set enable_thinking and max_new_tokens


In [39]:

import json
from pathlib import Path

source = Path("/content/fatima_fellowship_combined_25_reviewed.json")
destination = Path("/content/fatima_fellowship_uniform_rerun_inputs.json")

with source.open("r", encoding="utf-8") as f:
    cases = json.load(f)

assert len(cases) == 25, f"Expected 25 cases, found {len(cases)}"
assert len({c["case_id"] for c in cases}) == 25, "Duplicate case IDs found"

inputs = []

for case in cases:
    inputs.append({
        "case_id": case["case_id"],
        "topic": case.get("topic", ""),
        "original_prompt": case.get("prompt", ""),
        "reference_verdict": case.get("reference_verdict", ""),
        "reference_reason": case.get("reference_reason", "")
    })

missing_prompts = [
    item["case_id"] for item in inputs
    if not item["original_prompt"].strip()
]

if missing_prompts:
    raise ValueError(f"Missing prompts for: {missing_prompts}")

with destination.open("w", encoding="utf-8") as f:
    json.dump(inputs, f, ensure_ascii=False, indent=2)

print("Prepared cases:", len(inputs))
print("Unique case IDs:", len({x["case_id"] for x in inputs}))
print("Missing prompts:", len(missing_prompts))
print("Saved to:", destination)
print("\nFirst case ID:", inputs[0]["case_id"])
print("First prompt preview:\n", inputs[0]["original_prompt"][:500])


Prepared cases: 25
Unique case IDs: 25
Missing prompts: 0
Saved to: /content/fatima_fellowship_uniform_rerun_inputs.json

First case ID: case1
First prompt preview:
 For f_n(x)=x^n on [0,1], a student claims:
Pointwise convergence implies uniform convergence.

Reply in this exact format:
Verdict: CORRECT or INCORRECT
Reason: (maximum 3 sentences, with the key mathematical calculation)
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: (one sentence)


In [40]:

print("Model loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())

if "model" in globals():
    print("Model class:", type(model).__name__)
    print("Model device:", next(model.parameters()).device)

if "tokenizer" in globals():
    print("Tokenizer class:", type(tokenizer).__name__)
    print("Tokenizer has chat template:", bool(tokenizer.chat_template))


Model loaded: True
Tokenizer loaded: True
Model class: Qwen3ForCausalLM
Model device: cuda:0
Tokenizer class: Qwen2Tokenizer
Tokenizer has chat template: True


In [41]:

import json
import re
import torch
from pathlib import Path

input_path = Path("/content/fatima_fellowship_uniform_rerun_inputs.json")
output_path = Path("/content/fatima_fellowship_uniform_rerun_results.json")

with input_path.open("r", encoding="utf-8") as f:
    cases = json.load(f)

results = []

generation_settings = {
    "model_name": "Qwen/Qwen3-1.7B",
    "enable_thinking": False,
    "max_new_tokens": 180,
    "do_sample": False,
    "prompt_type": "uniform_followup",
}

for i, case in enumerate(cases, start=1):
    prompt = f"""Evaluate the mathematical claim and proof below.

{case["original_prompt"]}

Your answer MUST follow this exact format:
Verdict: CORRECT or INCORRECT
Reason: Give a mathematically valid explanation in at most 3 sentences.
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: Give one concise sentence.

Do not omit any field."""

    messages = [{"role": "user", "content": prompt}]

    rendered_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        rendered_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    verdict_match = re.search(
        r"Verdict:\s*(CORRECT|INCORRECT)\b",
        raw_output,
        re.IGNORECASE
    )
    confidence_match = re.search(
        r"Confidence:\s*(HIGH|MEDIUM|LOW)\b",
        raw_output,
        re.IGNORECASE
    )

    results.append({
        "case_id": case["case_id"],
        "topic": case.get("topic", ""),
        "reference_verdict": case.get("reference_verdict", ""),
        "original_prompt": case["original_prompt"],
        "uniform_prompt": prompt,
        "model_name": generation_settings["model_name"],
        "generation_settings": generation_settings,
        "raw_output": raw_output,
        "model_verdict": (
            verdict_match.group(1).upper()
            if verdict_match else "PARSE_FAILED"
        ),
        "verbal_confidence": (
            confidence_match.group(1).upper()
            if confidence_match else "NOT_PROVIDED"
        ),
        "verdict_correct": (
            verdict_match.group(1).upper()
            == case.get("reference_verdict", "").upper()
            if verdict_match and case.get("reference_verdict")
            else None
        ),
    })

    print(f"[{i}/{len(cases)}] {case['case_id']}: "
          f"{results[-1]['model_verdict']} | "
          f"Confidence: {results[-1]['verbal_confidence']}")

    # Save progress after each case so partial work is retained.
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "experiment": "uniform_followup",
                "generation_settings": generation_settings,
                "results": results
            },
            f,
            ensure_ascii=False,
            indent=2
        )

print("\nFinished.")
print("Cases processed:", len(results))
print("Verdict parse failures:", sum(
    r["model_verdict"] == "PARSE_FAILED" for r in results
))
print("Missing confidence:", sum(
    r["verbal_confidence"] == "NOT_PROVIDED" for r in results
))
print("Results saved to:", output_path)


[1/25] case1: INCORRECT | Confidence: MEDIUM
[2/25] case2: INCORRECT | Confidence: LOW
[3/25] case3: INCORRECT | Confidence: LOW
[4/25] case4: CORRECT | Confidence: HIGH
[5/25] case5: CORRECT | Confidence: HIGH
[6/25] case6: INCORRECT | Confidence: LOW
[7/25] case7: INCORRECT | Confidence: LOW
[8/25] case8: INCORRECT | Confidence: LOW
[9/25] case9: INCORRECT | Confidence: LOW
[10/25] case10: INCORRECT | Confidence: LOW
[11/25] case11: CORRECT | Confidence: HIGH
[12/25] case12: INCORRECT | Confidence: LOW
[13/25] case13: INCORRECT | Confidence: LOW
[14/25] case14: INCORRECT | Confidence: LOW
[15/25] case15: CORRECT | Confidence: HIGH
[16/25] case16: INCORRECT | Confidence: NOT_PROVIDED
[17/25] case17: INCORRECT | Confidence: LOW
[18/25] case18: INCORRECT | Confidence: LOW
[19/25] case19: INCORRECT | Confidence: LOW
[20/25] case20: INCORRECT | Confidence: LOW
[21/25] case21: INCORRECT | Confidence: LOW
[22/25] case22: INCORRECT | Confidence: LOW
[23/25] case23: INCORRECT | Confidence: LO

In [42]:

import json
import pandas as pd
from pathlib import Path

path = Path("/content/fatima_fellowship_uniform_rerun_results.json")

with path.open("r", encoding="utf-8") as f:
    saved = json.load(f)

results = saved["results"]
df = pd.DataFrame(results)

print("VERDICT ACCURACY BY CONFIDENCE")
print("=" * 50)

summary = df.groupby("verbal_confidence").agg(
    cases=("case_id", "count"),
    correct_verdicts=("verdict_correct", "sum")
)

summary["accuracy_percent"] = (
    100 * summary["correct_verdicts"] / summary["cases"]
).round(1)

print(summary.to_string())

print("\nOVERALL VERDICT ACCURACY")
print(f'{int(df["verdict_correct"].sum())}/{len(df)} '
      f'= {100 * df["verdict_correct"].mean():.1f}%')

print("\nCASES WITH MISSING CONFIDENCE")
missing = df[df["verbal_confidence"] == "NOT_PROVIDED"]

for _, row in missing.iterrows():
    print(f'\n--- {row["case_id"]} ---')
    print(row["raw_output"])

print("\nCONFIDENCE COVERAGE")
provided = (df["verbal_confidence"] != "NOT_PROVIDED").sum()
print(f"{provided}/{len(df)} = {100*provided/len(df):.1f}%")


VERDICT ACCURACY BY CONFIDENCE
                   cases  correct_verdicts  accuracy_percent
verbal_confidence                                           
HIGH                   5                 4              80.0
LOW                   17                11              64.7
MEDIUM                 1                 1             100.0
NOT_PROVIDED           2                 0               0.0

OVERALL VERDICT ACCURACY
16/25 = 64.0%

CASES WITH MISSING CONFIDENCE

--- case16 ---
Verdict: INCORRECT  
Reason: The proof is incorrect because while it is true that $ \frac{1}{n} \to 0 $ as $ n \to \infty $, the uniform convergence of $ f_n(x) = \frac{x^n}{n} $ to 0 on $[0,1]$ requires that the maximum of $ |f_n(x)| $ over $[0,1]$ tends to 0. However, the maximum occurs at $ x = 1 $, where $ |f_n(1)| = \frac{1}{n} $, which does tend to 0. The proof is technically correct in stating that $ |f_n(x)| \leq \frac{1}{n} $ for all $ x \in [0,1] $, but the claim that this implies uniform convergence 

In [43]:

import json
import re
import torch
from pathlib import Path

source = Path("/content/fatima_fellowship_uniform_rerun_results.json")
destination = Path("/content/fatima_fellowship_uniform_rerun_v2.json")

with source.open("r", encoding="utf-8") as f:
    previous = json.load(f)

new_results = []
settings = dict(previous["generation_settings"])
settings["max_new_tokens"] = 300
settings["version"] = "uniform_followup_v2"

for i, old in enumerate(previous["results"], start=1):
    messages = [{"role": "user", "content": old["uniform_prompt"]}]

    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(rendered, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    verdict_match = re.search(
        r"Verdict:\s*(CORRECT|INCORRECT)\b", raw, re.I
    )
    confidence_match = re.search(
        r"Confidence:\s*(HIGH|MEDIUM|LOW)\b", raw, re.I
    )

    verdict = verdict_match.group(1).upper() if verdict_match else "PARSE_FAILED"
    confidence = confidence_match.group(1).upper() if confidence_match else "NOT_PROVIDED"

    reference = old.get("reference_verdict", "")
    correct = (
        verdict == reference.upper()
        if verdict_match and reference
        else None
    )

    new_results.append({
        "case_id": old["case_id"],
        "topic": old.get("topic", ""),
        "reference_verdict": reference,
        "uniform_prompt": old["uniform_prompt"],
        "raw_output": raw,
        "model_verdict": verdict,
        "verbal_confidence": confidence,
        "verdict_correct": correct,
        "generation_settings": settings
    })

    with destination.open("w", encoding="utf-8") as f:
        json.dump({
            "experiment": "uniform_followup_v2",
            "generation_settings": settings,
            "results": new_results
        }, f, ensure_ascii=False, indent=2)

    print(
        f"[{i}/25] {old['case_id']}: {verdict} | "
        f"{confidence}"
    )

print("\n=== Summary ===")
print("Cases:", len(new_results))
print("Verdict parse failures:", sum(
    r["model_verdict"] == "PARSE_FAILED" for r in new_results
))
print("Missing confidence:", sum(
    r["verbal_confidence"] == "NOT_PROVIDED" for r in new_results
))
valid = [r for r in new_results if r["verdict_correct"] is not None]
print(
    "Verdict accuracy:",
    f"{sum(r['verdict_correct'] for r in valid)}/{len(valid)}",
    f"= {100*sum(r['verdict_correct'] for r in valid)/len(valid):.1f}%"
    if valid else ""
)
print("Saved to:", destination)


[1/25] case1: INCORRECT | MEDIUM
[2/25] case2: INCORRECT | LOW
[3/25] case3: INCORRECT | LOW
[4/25] case4: CORRECT | HIGH
[5/25] case5: CORRECT | HIGH
[6/25] case6: INCORRECT | LOW
[7/25] case7: INCORRECT | LOW
[8/25] case8: INCORRECT | LOW
[9/25] case9: INCORRECT | LOW
[10/25] case10: INCORRECT | LOW
[11/25] case11: CORRECT | HIGH
[12/25] case12: INCORRECT | LOW
[13/25] case13: INCORRECT | LOW
[14/25] case14: INCORRECT | LOW
[15/25] case15: CORRECT | HIGH
[16/25] case16: INCORRECT | MEDIUM
[17/25] case17: INCORRECT | LOW
[18/25] case18: INCORRECT | LOW
[19/25] case19: INCORRECT | LOW
[20/25] case20: INCORRECT | LOW
[21/25] case21: INCORRECT | LOW
[22/25] case22: INCORRECT | LOW
[23/25] case23: INCORRECT | LOW
[24/25] case24: INCORRECT | MEDIUM
[25/25] case25: CORRECT | HIGH

=== Summary ===
Cases: 25
Verdict parse failures: 0
Missing confidence: 0
Verdict accuracy: 16/25 = 64.0%
Saved to: /content/fatima_fellowship_uniform_rerun_v2.json


In [44]:

import json
import pandas as pd

path = "/content/fatima_fellowship_uniform_rerun_v2.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data["results"])

summary = df.groupby("verbal_confidence").agg(
    total_cases=("case_id", "count"),
    correct_verdicts=("verdict_correct", "sum")
)

summary["wrong_verdicts"] = (
    summary["total_cases"] - summary["correct_verdicts"]
)
summary["accuracy_percent"] = (
    100 * summary["correct_verdicts"] / summary["total_cases"]
).round(1)

print("CONFIDENCE VS VERDICT ACCURACY — V2")
print("=" * 55)
print(summary.to_string())

print("\nOverall accuracy:")
print(
    f"{int(df['verdict_correct'].sum())}/{len(df)} "
    f"= {100 * df['verdict_correct'].mean():.1f}%"
)

print("\nConfidence distribution:")
print(df["verbal_confidence"].value_counts().to_string())


CONFIDENCE VS VERDICT ACCURACY — V2
                   total_cases  correct_verdicts  wrong_verdicts  accuracy_percent
verbal_confidence                                                                 
HIGH                         5                 4               1              80.0
LOW                         17                11               6              64.7
MEDIUM                       3                 1               2              33.3

Overall accuracy:
16/25 = 64.0%

Confidence distribution:
verbal_confidence
LOW       17
HIGH       5
MEDIUM     3


In [45]:

import json
from pathlib import Path

path = Path("/content/fatima_fellowship_uniform_rerun_v2.json")

with path.open("r", encoding="utf-8") as f:
    data = json.load(f)

results = data["results"]

for case in results[:5]:
    print("\n" + "=" * 75)
    print("CASE:", case["case_id"])
    print("REFERENCE:", case["reference_verdict"])
    print("MODEL VERDICT:", case["model_verdict"])
    print("VERDICT CORRECT:", case["verdict_correct"])
    print("CONFIDENCE:", case["verbal_confidence"])
    print("\nMODEL REASONING:")
    print(case["raw_output"])



CASE: case1
REFERENCE: INCORRECT
MODEL VERDICT: INCORRECT
VERDICT CORRECT: True
CONFIDENCE: MEDIUM

MODEL REASONING:
Verdict: INCORRECT  
Reason: Pointwise convergence does not imply uniform convergence because the maximum difference between f_n(x) and f(x) does not necessarily approach zero uniformly across the interval [0, 1].  
Confidence: MEDIUM  
Why this confidence: The maximum of |x^n - x| on [0, 1] is 1/2 at x = 1/2, which does not approach zero as n increases.

CASE: case2
REFERENCE: INCORRECT
MODEL VERDICT: INCORRECT
VERDICT CORRECT: True
CONFIDENCE: LOW

MODEL REASONING:
Verdict: INCORRECT  
Reason: The proof incorrectly states that a sequence of functions converges uniformly to f on E because for every x in E, there exists N depending on x such that for all n ≥ N, |f_n(x) - f(x)| < ε. This is not a valid definition of uniform convergence; uniform convergence requires that for every ε > 0, there exists N such that for all x in E and all n ≥ N, |f_n(x) - f(x)| < ε.  
Confide

In [46]:

import json

path = "/content/fatima_fellowship_uniform_rerun_v2.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

for case in data["results"][5:10]:
    print("\n" + "=" * 75)
    print("CASE:", case["case_id"])
    print("REFERENCE:", case["reference_verdict"])
    print("MODEL VERDICT:", case["model_verdict"])
    print("VERDICT CORRECT:", case["verdict_correct"])
    print("CONFIDENCE:", case["verbal_confidence"])
    print("\nMODEL REASONING:")
    print(case["raw_output"])



CASE: case6
REFERENCE: INCORRECT
MODEL VERDICT: INCORRECT
VERDICT CORRECT: True
CONFIDENCE: LOW

MODEL REASONING:
Verdict: INCORRECT  
Reason: The student's conclusion is incorrect because the convergence of $ f_n(x) = x^n $ to 0 pointwise on [0, 1) does not imply uniform convergence.  
Confidence: LOW  
Why this confidence: Uniform convergence requires the maximum of $ |f_n(x) - f(x)| $ to converge to 0, which is not satisfied here.

CASE: case7
REFERENCE: INCORRECT
MODEL VERDICT: INCORRECT
VERDICT CORRECT: True
CONFIDENCE: LOW

MODEL REASONING:
Verdict: INCORRECT  
Reason: The student incorrectly assumes uniform convergence based on pointwise convergence. Uniform convergence requires the maximum of |f_n(x) - f(x)| to tend to zero, but the maximum of |sin(x/n)| is 1, which does not tend to zero.  
Confidence: LOW  
Why this confidence: The maximum value of |sin(x/n)| is 1, so the uniform convergence claim is false.

CASE: case8
REFERENCE: INCORRECT
MODEL VERDICT: INCORRECT
VERDICT CO

In [47]:

import json

path = "/content/fatima_fellowship_uniform_rerun_v2.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

for case in data["results"][10:15]:
    print("\n" + "=" * 75)
    print("CASE:", case["case_id"])
    print("REFERENCE:", case["reference_verdict"])
    print("MODEL VERDICT:", case["model_verdict"])
    print("VERDICT CORRECT:", case["verdict_correct"])
    print("CONFIDENCE:", case["verbal_confidence"])
    print("\nMODEL REASONING:")
    print(case["raw_output"])



CASE: case11
REFERENCE: CORRECT
MODEL VERDICT: CORRECT
VERDICT CORRECT: True
CONFIDENCE: HIGH

MODEL REASONING:
Verdict: CORRECT  
Reason: The supremum of |f_n(x) - f(x)| is 1/n, which tends to 0 as n → ∞, so the convergence is uniform.  
Confidence: HIGH  
Why this confidence: The uniform convergence is established by the uniform bound on the difference.

CASE: case12
REFERENCE: CORRECT
MODEL VERDICT: INCORRECT
VERDICT CORRECT: False
CONFIDENCE: LOW

MODEL REASONING:
Verdict: INCORRECT  
Reason: The inequality $ n + x^2 \geq 2\sqrt{n}|x| $ is valid, but it does not imply uniform convergence. The bound $ \frac{1}{2\sqrt{n}} $ is not uniform across all $ x $, as it depends on $ n $ and $ x $, and the convergence is not uniform.  
Confidence: LOW  
Why this confidence: The bound is not uniform, and the inequality does not guarantee uniform convergence.

CASE: case13
REFERENCE: INCORRECT
MODEL VERDICT: INCORRECT
VERDICT CORRECT: True
CONFIDENCE: LOW

MODEL REASONING:
Verdict: INCORRECT  

In [48]:
import json

path = "/content/fatima_fellowship_uniform_rerun_v2.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

for case in data["results"][10:15]:
    print("\n" + "=" * 75)
    print("CASE:", case["case_id"])
    print("ORIGINAL PROMPT:")
    print(case.get("original_prompt", "Original prompt field not found"))


CASE: case11
ORIGINAL PROMPT:
Original prompt field not found

CASE: case12
ORIGINAL PROMPT:
Original prompt field not found

CASE: case13
ORIGINAL PROMPT:
Original prompt field not found

CASE: case14
ORIGINAL PROMPT:
Original prompt field not found

CASE: case15
ORIGINAL PROMPT:
Original prompt field not found


In [49]:
import json

path = "/content/fatima_fellowship_uniform_rerun_v2.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Top-level keys:", list(data.keys()))
print("\nKeys inside the first result:")
print(list(data["results"][0].keys()))

print("\nFull first result:")
print(json.dumps(data["results"][0], ensure_ascii=False, indent=2)[:6000])

Top-level keys: ['experiment', 'generation_settings', 'results']

Keys inside the first result:
['case_id', 'topic', 'reference_verdict', 'uniform_prompt', 'raw_output', 'model_verdict', 'verbal_confidence', 'verdict_correct', 'generation_settings']

Full first result:
{
  "case_id": "case1",
  "topic": "Pointwise versus uniform convergence on [0,1]",
  "reference_verdict": "INCORRECT",
  "uniform_prompt": "Evaluate the mathematical claim and proof below.\n\nFor f_n(x)=x^n on [0,1], a student claims:\nPointwise convergence implies uniform convergence.\n\nReply in this exact format:\nVerdict: CORRECT or INCORRECT\nReason: (maximum 3 sentences, with the key mathematical calculation)\nConfidence: HIGH, MEDIUM, or LOW\nWhy this confidence: (one sentence)\n\nYour answer MUST follow this exact format:\nVerdict: CORRECT or INCORRECT\nReason: Give a mathematically valid explanation in at most 3 sentences.\nConfidence: HIGH, MEDIUM, or LOW\nWhy this confidence: Give one concise sentence.\n\nDo 

In [50]:
import json

path = "/content/fatima_fellowship_uniform_rerun_v2.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

for case in data["results"][10:15]:
    print("\n" + "=" * 75)
    print("CASE:", case["case_id"])
    print("TOPIC:", case["topic"])
    print("REFERENCE:", case["reference_verdict"])
    print("MODEL:", case["model_verdict"])
    print("CONFIDENCE:", case["verbal_confidence"])
    print("\nORIGINAL QUESTION AND PROMPT:")
    print(case["uniform_prompt"])


CASE: case11
TOPIC: Constant functions on an arbitrary nonempty domain
REFERENCE: CORRECT
MODEL: CORRECT
CONFIDENCE: HIGH

ORIGINAL QUESTION AND PROMPT:
Evaluate the mathematical claim and proof below.

Let E be any nonempty set and define f_n(x)=1/n and f(x)=0 for all x in E.
A student claims that f_n converges uniformly to f on E because
sup over x in E of |f_n(x)-f(x)| equals 1/n, which tends to zero.
Is the claim correct? Explain briefly.
Reply with:
Verdict: CORRECT or INCORRECT
Reason: maximum 2 sentences
Confidence: HIGH, MEDIUM, or LOW

Your answer MUST follow this exact format:
Verdict: CORRECT or INCORRECT
Reason: Give a mathematically valid explanation in at most 3 sentences.
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: Give one concise sentence.

Do not omit any field.

CASE: case12
TOPIC: Uniform convergence using an algebraic bound
REFERENCE: CORRECT
MODEL: INCORRECT
CONFIDENCE: LOW

ORIGINAL QUESTION AND PROMPT:
Evaluate the mathematical claim and proof below.


In [51]:
import json

path = "/content/fatima_fellowship_uniform_rerun_v2.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

for case in data["results"][15:20]:
    print("\n" + "=" * 75)
    print("CASE:", case["case_id"])
    print("TOPIC:", case["topic"])
    print("REFERENCE:", case["reference_verdict"])
    print("MODEL VERDICT:", case["model_verdict"])
    print("VERDICT CORRECT:", case["verdict_correct"])
    print("CONFIDENCE:", case["verbal_confidence"])
    print("\nORIGINAL QUESTION AND PROMPT:")
    print(case["uniform_prompt"])
    print("\nMODEL REASONING:")
    print(case["raw_output"])


CASE: case16
TOPIC: uniform_bound
REFERENCE: CORRECT
MODEL VERDICT: INCORRECT
VERDICT CORRECT: False
CONFIDENCE: MEDIUM

ORIGINAL QUESTION AND PROMPT:
Evaluate the mathematical claim and proof below.


Determine whether the following claim and proof are correct.

Claim: f_n(x) = x^n/n converges uniformly to 0 on [0,1].

Proof: For every x in [0,1], |f_n(x)| = x^n/n <= 1/n.
Since 1/n tends to 0 independently of x, the convergence is uniform.

Evaluate the proof, not just the claim.


Your answer MUST follow this exact format:
Verdict: CORRECT or INCORRECT
Reason: Give a mathematically valid explanation in at most 3 sentences.
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: Give one concise sentence.

Do not omit any field.

MODEL REASONING:
Verdict: INCORRECT  
Reason: The proof is incorrect because while it is true that $ \frac{1}{n} \to 0 $ as $ n \to \infty $, the uniform convergence of $ f_n(x) = \frac{x^n}{n} $ to 0 on $[0,1]$ requires that the maximum of $ |f_n(x)| $ over $

In [53]:
import json

path = "/content/fatima_fellowship_uniform_rerun_v2.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

for case in data["results"][20:25]:
    print("\n" + "=" * 75)
    print("CASE:", case["case_id"])
    print("TOPIC:", case["topic"])
    print("REFERENCE:", case["reference_verdict"])
    print("MODEL VERDICT:", case["model_verdict"])
    print("VERDICT CORRECT:", case["verdict_correct"])
    print("CONFIDENCE:", case["verbal_confidence"])
    print("\nORIGINAL QUESTION AND PROMPT:")
    print(case["uniform_prompt"])
    print("\nMODEL REASONING:")
    print(case["raw_output"])


CASE: case21
TOPIC: compactness_and_continuity
REFERENCE: INCORRECT
MODEL VERDICT: INCORRECT
VERDICT CORRECT: True
CONFIDENCE: LOW

ORIGINAL QUESTION AND PROMPT:
Evaluate the mathematical claim and proof below.


Determine whether the following mathematical statement is correct.

Statement: If continuous functions f_n on [0,1] converge pointwise
to a continuous function f, then f_n necessarily converge uniformly
to f, because [0,1] is compact.

Give a verdict and justify it mathematically.


Your answer MUST follow this exact format:
Verdict: CORRECT or INCORRECT
Reason: Give a mathematically valid explanation in at most 3 sentences.
Confidence: HIGH, MEDIUM, or LOW
Why this confidence: Give one concise sentence.

Do not omit any field.

MODEL REASONING:
Verdict: INCORRECT  
Reason: The statement is incorrect because pointwise convergence to a continuous function does not imply uniform convergence, even on a compact interval like [0,1]. Uniform convergence requires stronger conditions